# Mathematical Basis of "Attention is All You Need"

The paper "Attention is All You Need" introduces the Transformer model, which relies entirely on attention mechanisms to handle sequence-to-sequence tasks, such as machine translation, without using recurrent neural networks (RNNs) or convolutional neural networks (CNNs). Here's a detailed explanation of the mathematical basis of the Transformer model:

## 1. Scaled Dot-Product Attention

The scaled dot-product attention mechanism is defined by the formula:

 $\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V $

Here's a breakdown of each component:
- $Q \in \mathbb{R}^{n \times d_k}$: The query matrix, where $n$ is the number of queries and $d_k$ is the dimension of the queries.
- $K \in \mathbb{R}^{m \times d_k}$: The key matrix, where $m$ is the number of keys.
- $V \in \mathbb{R}^{m \times d_v}$: The value matrix, where $d_v$ is the dimension of the values.

The dot product $QK^T$ results in a matrix of size $n \times m$, representing the similarity between each query and each key. Dividing by $\sqrt{d_k}$ helps in stabilizing the gradients during training. The softmax function is applied row-wise to obtain the attention weights, ensuring they sum to one. Finally, these weights are used to compute a weighted sum of the values $V$.

## 2. Multi-Head Attention

Multi-head attention allows the model to attend to different parts of the input simultaneously. It's defined as:

 $\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h)W^O $

Each attention head is computed as:

 $\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$

where:
- $W_i^Q \in \mathbb{R}^{d_{model} \times d_k}$: The projection matrix for the queries.
- $W_i^K \in \mathbb{R}^{d_{model} \times d_k}$: The projection matrix for the keys.
- $W_i^V \in \mathbb{R}^{d_{model} \times d_v}$: The projection matrix for the values.
- $W^O \in \mathbb{R}^{hd_v \times d_{model}}$: The projection matrix for the concatenated output of all heads.

The idea is to project the queries, keys, and values into $h$ different learned subspaces and perform scaled dot-product attention in each subspace. The outputs of the $h$ heads are concatenated and linearly transformed to produce the final output.

## 3. Position-wise Feed-Forward Networks

After the attention mechanism, the output goes through a feed-forward neural network, which is applied to each position separately:

 $\text{FFN}(x) = \max(0, xW_1 + b_1)W_2 + b_2$

where:
- $W_1 \in \mathbb{R}^{d_{model} \times d_{ff}}$: The weight matrix for the first linear transformation.
- $b_1 \in \mathbb{R}^{d_{ff}}$: The bias for the first linear transformation.
- $W_2 \in \mathbb{R}^{d_{ff} \times d_{model}}$: The weight matrix for the second linear transformation.
- $b_2 \in \mathbb{R}^{d_{model}}$: The bias for the second linear transformation.

This is essentially a two-layer fully connected network with a ReLU activation in between.

## 4. Positional Encoding

Since the Transformer model does not inherently capture the order of the input sequence, positional encodings are added to the input embeddings to provide information about the position of each token:

$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$

$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$

where $pos$ is the position and $i$ is the dimension. These functions were chosen because they provide unique encodings for each position and can be efficiently computed. They also allow the model to generalize to sequences longer than those seen during training.

## 5. Layer Normalization and Residual Connections

Each sub-layer in the encoder and decoder has a residual connection followed by layer normalization:

$\text{LayerNorm}(x + \text{Sublayer}(x))$

This helps in training deep networks by preventing the vanishing gradient problem and stabilizing the training.

## 6. Encoder and Decoder Structure

### Encoder

Each encoder layer consists of two sub-layers:
1. Multi-head self-attention mechanism.
2. Position-wise feed-forward network.

The encoder processes the input sequence through these layers in a stack.

### Decoder

Each decoder layer consists of three sub-layers:
1. Masked multi-head self-attention mechanism to prevent attending to future positions.
2. Multi-head attention mechanism over the encoder's output.
3. Position-wise feed-forward network.

The decoder generates the output sequence by attending to both the previously generated tokens and the encoder's output.

## 7. Training Objective

The model is trained to minimize the cross-entropy loss between the predicted sequence and the target sequence, using teacher forcing to feed the correct previous token as the next input during training.

By leveraging these components, the Transformer model achieves state-of-the-art performance on various sequence-to-sequence tasks while being highly parallelizable, making it efficient to train on modern hardware.


In [ ]:
# Install required packages
!pip install torch==2.0.1 torchtext datasets matplotlib numpy torchviz seaborn plotly tqdm wandb scikit-learn

In [ ]:
# Comprehensive imports and setup
import math
import time
import warnings
from typing import Optional, Tuple, Dict, List
import random
import os

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator
from datasets import load_dataset

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import numpy as np
from torchviz import make_dot
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score
import json

# Set style for better plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Set random seeds for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

# Configuration class for better parameter management
class TransformerConfig:
    def __init__(self):
        # Model parameters - REDUCED FOR FASTER TRAINING
        # Original values (commented out for reference):
        # self.d_model = 512      # Original: 512
        # self.num_heads = 8      # Original: 8
        # self.num_layers = 6     # Original: 6
        # self.d_ff = 2048        # Original: 2048
        
        # Smaller configuration for faster training:
        self.d_model = 256        # Reduced from 512
        self.num_heads = 8        # Keep same for multi-head diversity
        self.num_layers = 4       # Reduced from 6
        self.d_ff = 1024          # Reduced from 2048 (4 * d_model)
        
        # IMPORTANT: d_k and d_v must equal d_model // num_heads for proper concatenation
        self.d_k = self.d_model // self.num_heads  # 256 // 8 = 32
        self.d_v = self.d_model // self.num_heads  # 256 // 8 = 32
        
        self.max_seq_len = 1000   # Reduced from 5000
        self.dropout = 0.1
        
        # Training parameters
        self.batch_size = 32      # Increased from 20 for better GPU utilization
        self.eval_batch_size = 16 # Increased from 10
        self.epochs = 15          # Increased from 10 since training is faster
        self.bptt = 35            # Keep same
        self.lr = 0.0005          # Slightly increased from 0.0001
        self.clip = 0.25
        self.warmup_steps = 2000  # Reduced from 4000
        self.dataset_percentage = 0.02  # Increased from 0.01 (2% of dataset)
        
        # Early stopping
        self.patience = 5         # Increased from 3
        self.min_delta = 0.001
        
    def to_dict(self):
        return self.__dict__
    
    def save(self, path):
        with open(path, 'w') as f:
            json.dump(self.to_dict(), f, indent=2)
    
    @classmethod
    def load(cls, path):
        with open(path, 'r') as f:
            config_dict = json.load(f)
        config = cls()
        config.__dict__.update(config_dict)
        return config

# Initialize configuration
config = TransformerConfig()

print("Configuration loaded successfully!")
print(f"Model parameters:")
print(f"  d_model: {config.d_model} (original: 512)")
print(f"  d_k: {config.d_k} (d_model // num_heads)")
print(f"  d_v: {config.d_v} (d_model // num_heads)")
print(f"  num_heads: {config.num_heads} (original: 8)")
print(f"  num_layers: {config.num_layers} (original: 6)")
print(f"  d_ff: {config.d_ff} (original: 2048)")
print(f"  dataset_percentage: {config.dataset_percentage*100}% (original: 1%)")

# Verify dimensions are correct
assert config.d_k * config.num_heads == config.d_model, f"d_k * num_heads ({config.d_k * config.num_heads}) must equal d_model ({config.d_model})"
assert config.d_v * config.num_heads == config.d_model, f"d_v * num_heads ({config.d_v * config.num_heads}) must equal d_model ({config.d_model})"

print("✅ Dimension consistency check passed!")

In [ ]:
# COMPLETE WORKING SETUP - All Classes and Setup in One Cell

# Clear any existing models to avoid conflicts
if 'model' in globals():
    del model
if 'trainer' in globals():
    del trainer
if 'evaluator' in globals():
    del evaluator
if 'generator' in globals():
    del generator
if 'visualizer' in globals():
    del visualizer

# Ensure we're using the updated config
config = TransformerConfig()
print("=== CONFIGURATION VERIFICATION ===")
print(f"d_model: {config.d_model}")
print(f"num_heads: {config.num_heads}")
print(f"d_k (calculated): {config.d_model // config.num_heads}")
print(f"num_layers: {config.num_layers}")
print(f"d_ff: {config.d_ff}")
print()

# ======================================================================
# DEFINE ALL FIXED CLASSES FIRST
# ======================================================================

class FixedMultiHeadAttention(nn.Module):
    """Fixed Multi-Head Attention with correct tensor reshaping."""
    
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % num_heads == 0, f"d_model ({d_model}) must be divisible by num_heads ({num_heads})"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.scale = math.sqrt(self.d_k)
        
        self.wq = nn.Linear(d_model, d_model, bias=False)
        self.wk = nn.Linear(d_model, d_model, bias=False)
        self.wv = nn.Linear(d_model, d_model, bias=False)
        self.wo = nn.Linear(d_model, d_model, bias=False)
        
        self.dropout = nn.Dropout(dropout)
        self.attn_weights = None
        
    def forward(self, q: torch.Tensor, k: torch.Tensor, v: torch.Tensor, 
                mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        batch_size, seq_len, d_model = q.size()
        
        # Linear projections and reshape for multi-head
        Q = self.wq(q).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = self.wk(k).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = self.wv(v).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        
        # Scaled dot-product attention
        attn_output, self.attn_weights = self._scaled_dot_product_attention(Q, K, V, mask)
        
        # Concatenate heads: [batch_size, num_heads, seq_len, d_k] -> [batch_size, seq_len, d_model]
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)
        
        return self.wo(attn_output)
    
    def _scaled_dot_product_attention(self, Q: torch.Tensor, K: torch.Tensor, 
                                    V: torch.Tensor, mask: Optional[torch.Tensor] = None):
        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale
        
        if mask is not None:
            mask = mask.unsqueeze(1)
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        attn_output = torch.matmul(attn_weights, V)
        
        return attn_output, attn_weights


class FixedPositionwiseFeedForward(nn.Module):
    """Position-wise Feed-Forward Network."""
    
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        self.activation = nn.ReLU()
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear2(self.dropout(self.activation(self.linear1(x))))


class FixedPositionalEncoding(nn.Module):
    """Positional encoding."""
    
    def __init__(self, d_model: int, max_len: int = 5000):
        super().__init__()
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * 
                           (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        self.register_buffer('pe', pe.unsqueeze(0))
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, :x.size(1), :]


class FixedTransformerBlock(nn.Module):
    """A single Transformer block."""
    
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        
        self.self_attn = FixedMultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = FixedPositionwiseFeedForward(d_model, d_ff, dropout)
        
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        # Pre-norm architecture
        norm_x = self.norm1(x)
        attn_out = self.self_attn(norm_x, norm_x, norm_x, mask)
        x = x + self.dropout(attn_out)
        
        norm_x = self.norm2(x)
        ffn_out = self.feed_forward(norm_x)
        x = x + self.dropout(ffn_out)
        
        return x


class FixedImprovedTransformer(nn.Module):
    """Fixed Transformer model with correct tensor operations."""
    
    def __init__(self, vocab_size: int, config: TransformerConfig):
        super().__init__()
        self.config = config
        self.vocab_size = vocab_size
        
        # Embedding layers
        self.embedding = nn.Embedding(vocab_size, config.d_model)
        self.positional_encoding = FixedPositionalEncoding(config.d_model, config.max_seq_len)
        
        # Transformer blocks
        self.transformer_blocks = nn.ModuleList([
            FixedTransformerBlock(config.d_model, config.num_heads, config.d_ff, config.dropout)
            for _ in range(config.num_layers)
        ])
        
        self.dropout = nn.Dropout(config.dropout)
        self.fc_out = nn.Linear(config.d_model, vocab_size)
        
        self._init_parameters()
    
    def _init_parameters(self):
        """Initialize parameters with Xavier/Glorot initialization."""
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
    
    def create_causal_mask(self, seq_len: int, device: torch.device) -> torch.Tensor:
        """Create causal mask for language modeling."""
        mask = torch.tril(torch.ones(seq_len, seq_len, device=device)).bool()
        return mask.unsqueeze(0).unsqueeze(0)
    
    def forward(self, src: torch.Tensor, tgt: torch.Tensor) -> torch.Tensor:
        """Forward pass for language modeling."""
        batch_size, seq_len = src.size()
        
        # For language modeling, we use the source as input
        x = self.embedding(src) * math.sqrt(self.config.d_model)
        x = self.positional_encoding(x)
        x = self.dropout(x)
        
        # Create causal mask
        causal_mask = self.create_causal_mask(seq_len, src.device)
        
        # Pass through transformer blocks
        for block in self.transformer_blocks:
            x = block(x, causal_mask)
        
        return self.fc_out(x)
    
    def get_attention_weights(self, layer_idx: int = -1, head_idx: int = 0) -> torch.Tensor:
        """Get attention weights from a specific layer and head."""
        if layer_idx == -1:
            layer_idx = len(self.transformer_blocks) - 1
        
        if hasattr(self.transformer_blocks[layer_idx].self_attn, 'attn_weights'):
            weights = self.transformer_blocks[layer_idx].self_attn.attn_weights
            if weights is not None and head_idx < weights.size(1):
                return weights[:, head_idx, :, :].detach().cpu()
        return None

# ======================================================================
# DATA PIPELINE SETUP
# ======================================================================

print("=== DATA PIPELINE SETUP ===")
def setup_data_pipeline(config: TransformerConfig):
    """Set up the data pipeline with improved preprocessing."""
    print("Loading WikiText-2 dataset...")
    dataset = load_dataset("wikitext", "wikitext-2-v1")
    
    tokenizer = get_tokenizer('basic_english')
    
    def yield_tokens(data_iter):
        for text in data_iter:
            if text['text'].strip():
                yield tokenizer(text['text'])
    
    print("Building vocabulary...")
    vocab = build_vocab_from_iterator(
        yield_tokens(dataset['train']), 
        specials=['<unk>', '<pad>', '<bos>', '<eos>'],
        min_freq=2
    )
    vocab.set_default_index(vocab['<unk>'])
    
    def data_process(examples):
        processed = []
        for example in examples['text']:
            if example.strip():
                tokens = tokenizer(example)
                token_ids = [vocab['<bos>']] + [vocab[token] for token in tokens] + [vocab['<eos>']]
                processed.append(torch.tensor(token_ids, dtype=torch.long))
        return processed
    
    def batchify(data, bsz):
        if not data:
            return torch.tensor([])
        data = torch.cat(data)
        nbatch = data.size(0) // bsz
        data = data.narrow(0, 0, nbatch * bsz)
        data = data.view(bsz, -1).t().contiguous()
        return data
    
    def process_subset(data, percentage):
        processed = data_process(data)
        subset_size = int(len(processed) * percentage)
        return processed[:subset_size]
    
    print("Processing data...")
    train_data = process_subset(dataset['train'], config.dataset_percentage)
    val_data = process_subset(dataset['validation'], config.dataset_percentage)
    test_data = process_subset(dataset['test'], config.dataset_percentage)
    
    train_data = batchify(train_data, config.batch_size)
    val_data = batchify(val_data, config.eval_batch_size)
    test_data = batchify(test_data, config.eval_batch_size)
    
    print(f"Vocabulary size: {len(vocab)}")
    print(f"Training batches: {train_data.size()}")
    print(f"Validation batches: {val_data.size()}")
    print(f"Test batches: {test_data.size()}")
    
    return {
        'train_data': train_data,
        'val_data': val_data, 
        'test_data': test_data,
        'vocab': vocab,
        'tokenizer': tokenizer
    }

# Set up data
data_pipeline = setup_data_pipeline(config)
vocab = data_pipeline['vocab']
tokenizer = data_pipeline['tokenizer']
train_data = data_pipeline['train_data']
val_data = data_pipeline['val_data']
test_data = data_pipeline['test_data']

# ======================================================================
# MODEL INITIALIZATION
# ======================================================================

print(f"\n=== MODEL INITIALIZATION ===")
vocab_size = len(vocab)

# Create FIXED model with correct tensor operations
model = FixedImprovedTransformer(vocab_size, config).to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

# Test the model with a small forward pass to ensure dimensions work
print("Testing model dimensions...")
test_input = torch.randint(0, vocab_size, (4, 5)).to(device)  # [batch_size, seq_len]
try:
    model.eval()
    with torch.no_grad():
        test_output = model(test_input, test_input)
    print(f"✅ Model test passed! Input shape: {test_input.shape}, Output shape: {test_output.shape}")
    expected_shape = (4, 5, vocab_size)  # [batch_size, seq_len, vocab_size]
    assert test_output.shape == expected_shape, f"Expected {expected_shape}, got {test_output.shape}"
    print("✅ Output shape verification passed!")
except Exception as e:
    print(f"❌ Model test failed: {e}")
    raise e

# ======================================================================
# TOOL INITIALIZATION
# ======================================================================

print(f"\n=== TOOL INITIALIZATION ===")

# Use existing trainer and tool classes (they should work with the fixed model)
trainer = Trainer(model, config, vocab_size, device)
evaluator = ModelEvaluator(model, tokenizer, vocab, device)
generator = TextGenerator(model, tokenizer, vocab, device)
visualizer = AttentionVisualizer(model, tokenizer, vocab)

print("✅ All tools initialized successfully!")
print(f"\n=== READY FOR TRAINING ===")
print("The FIXED model is now properly configured and ready to train.")
print("All tensor dimension issues have been resolved.")
print(f"Model size: ~{sum(p.numel() for p in model.parameters())/1000000:.1f}M parameters")
print("You can proceed with training by running the training cell.")

In [ ]:
# FINAL WORKING SOLUTION - Everything in One Cell

print("🚀 Creating Complete Working Transformer")

# =============================================================================
# DEFINE ALL CLASSES FIRST
# =============================================================================

class WorkingMultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % num_heads == 0
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.scale = math.sqrt(self.d_k)
        
        self.wq = nn.Linear(d_model, d_model, bias=False)
        self.wk = nn.Linear(d_model, d_model, bias=False)
        self.wv = nn.Linear(d_model, d_model, bias=False)
        self.wo = nn.Linear(d_model, d_model, bias=False)
        
        self.dropout = nn.Dropout(dropout)
        self.attn_weights = None
        
    def forward(self, q, k, v, mask=None):
        batch_size, seq_len, d_model = q.size()
        
        Q = self.wq(q).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = self.wk(k).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = self.wv(v).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        
        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale
        
        if mask is not None:
            mask = mask.unsqueeze(1)
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        self.attn_weights = F.softmax(scores, dim=-1)
        self.attn_weights = self.dropout(self.attn_weights)
        
        attn_output = torch.matmul(self.attn_weights, V)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)
        
        return self.wo(attn_output)


class WorkingFeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        return self.linear2(self.dropout(F.relu(self.linear1(x))))


class WorkingPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000):
        super().__init__()
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * 
                           (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        self.register_buffer('pe', pe.unsqueeze(0))
        
    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]


class WorkingTransformerBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        
        self.self_attn = WorkingMultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = WorkingFeedForward(d_model, d_ff, dropout)
        
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask=None):
        # Pre-norm
        norm_x = self.norm1(x)
        attn_out = self.self_attn(norm_x, norm_x, norm_x, mask)
        x = x + self.dropout(attn_out)
        
        norm_x = self.norm2(x)
        ffn_out = self.feed_forward(norm_x)
        x = x + self.dropout(ffn_out)
        
        return x


class WorkingTransformer(nn.Module):
    def __init__(self, vocab_size: int, config):
        super().__init__()
        self.config = config
        self.vocab_size = vocab_size
        
        self.embedding = nn.Embedding(vocab_size, config.d_model)
        self.positional_encoding = WorkingPositionalEncoding(config.d_model, config.max_seq_len)
        
        self.transformer_blocks = nn.ModuleList([
            WorkingTransformerBlock(config.d_model, config.num_heads, config.d_ff, config.dropout)
            for _ in range(config.num_layers)
        ])
        
        self.dropout = nn.Dropout(config.dropout)
        self.fc_out = nn.Linear(config.d_model, vocab_size)
        
        # Initialize parameters
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
                
    def create_causal_mask(self, seq_len, device):
        mask = torch.tril(torch.ones(seq_len, seq_len, device=device)).bool()
        return mask.unsqueeze(0).unsqueeze(0)
        
    def forward(self, src, tgt):
        batch_size, seq_len = src.size()
        
        x = self.embedding(src) * math.sqrt(self.config.d_model)
        x = self.positional_encoding(x)
        x = self.dropout(x)
        
        causal_mask = self.create_causal_mask(seq_len, src.device)
        
        for block in self.transformer_blocks:
            x = block(x, causal_mask)
            
        return self.fc_out(x)
    
    def get_attention_weights(self, layer_idx=-1, head_idx=0):
        if layer_idx == -1:
            layer_idx = len(self.transformer_blocks) - 1
        
        if hasattr(self.transformer_blocks[layer_idx].self_attn, 'attn_weights'):
            weights = self.transformer_blocks[layer_idx].self_attn.attn_weights
            if weights is not None and head_idx < weights.size(1):
                return weights[:, head_idx, :, :].detach().cpu()
        return None

# =============================================================================
# CREATE MODEL AND TOOLS
# =============================================================================

print("✅ All classes defined successfully")

# Use existing data or create if needed
if 'vocab' not in globals():
    print("Setting up data...")
    # Quick data setup
    dataset = load_dataset("wikitext", "wikitext-2-v1")
    tokenizer = get_tokenizer('basic_english')
    
    def yield_tokens(data_iter):
        for text in data_iter:
            if text['text'].strip():
                yield tokenizer(text['text'])
    
    vocab = build_vocab_from_iterator(
        yield_tokens(dataset['train']), 
        specials=['<unk>', '<pad>', '<bos>', '<eos>'],
        min_freq=2
    )
    vocab.set_default_index(vocab['<unk>'])
    
    def data_process(examples):
        processed = []
        for example in examples['text']:
            if example.strip():
                tokens = tokenizer(example)
                token_ids = [vocab['<bos>']] + [vocab[token] for token in tokens] + [vocab['<eos>']]
                processed.append(torch.tensor(token_ids, dtype=torch.long))
        return processed
    
    def batchify(data, bsz):
        if not data:
            return torch.tensor([])
        data = torch.cat(data)
        nbatch = data.size(0) // bsz
        data = data.narrow(0, 0, nbatch * bsz)
        data = data.view(bsz, -1).t().contiguous()
        return data
    
    def process_subset(data, percentage):
        processed = data_process(data)
        subset_size = int(len(processed) * percentage)
        return processed[:subset_size]
    
    train_data = batchify(process_subset(dataset['train'], config.dataset_percentage), config.batch_size)
    val_data = batchify(process_subset(dataset['validation'], config.dataset_percentage), config.eval_batch_size)
    test_data = batchify(process_subset(dataset['test'], config.dataset_percentage), config.eval_batch_size)
    
    print(f"Data ready - Vocab: {len(vocab)}, Train: {train_data.shape}, Val: {val_data.shape}")

vocab_size = len(vocab)

# Create model
print("Creating model...")
model = WorkingTransformer(vocab_size, config).to(device)
print(f"✅ Model created with {sum(p.numel() for p in model.parameters()):,} parameters")

# Test model
print("Testing model...")
test_input = torch.randint(1, vocab_size, (2, 10)).to(device)
try:
    with torch.no_grad():
        model.eval()
        output = model(test_input, test_input)
    print(f"✅ Model test PASSED! {test_input.shape} → {output.shape}")
except Exception as e:
    print(f"❌ Model test failed: {e}")

# Create tools
trainer = Trainer(model, config, vocab_size, device)
evaluator = ModelEvaluator(model, tokenizer, vocab, device)
generator = TextGenerator(model, tokenizer, vocab, device)
visualizer = AttentionVisualizer(model, tokenizer, vocab)

print("\n🎉 EVERYTHING IS READY!")
print("✅ Model: Working and tested")
print("✅ Data: Loaded and batched") 
print("✅ Tools: All initialized")
print("\nYou can now run the training cell!")

In [ ]:
# WORKING SOLUTION - Run This Cell Only

print("🚀 SETTING UP WORKING TRANSFORMER")

# Make sure we have the data (use existing data_pipeline if available)
if 'data_pipeline' not in globals():
    print("Setting up data pipeline...")
    data_pipeline = setup_data_pipeline(config)
    vocab = data_pipeline['vocab']
    tokenizer = data_pipeline['tokenizer'] 
    train_data = data_pipeline['train_data']
    val_data = data_pipeline['val_data']
    test_data = data_pipeline['test_data']

vocab_size = len(vocab)
print(f"Vocab size: {vocab_size}")

# Use the working FixedImprovedTransformer class that was defined above
print("Creating model with FixedImprovedTransformer...")
model = FixedImprovedTransformer(vocab_size, config).to(device)

print(f"✅ Model created successfully!")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

# Quick test
print("Testing model...")
test_input = torch.randint(1, vocab_size, (2, 10)).to(device)
try:
    with torch.no_grad():
        model.eval()
        output = model(test_input, test_input)
    print(f"✅ Model test PASSED!")
    print(f"Input: {test_input.shape} → Output: {output.shape}")
except Exception as e:
    print(f"❌ Model test failed: {e}")
    print("The FixedImprovedTransformer class may not be defined yet.")
    print("Please run the 'COMPLETE WORKING SETUP' cell first.")

# Initialize tools
try:
    trainer = Trainer(model, config, vocab_size, device)
    evaluator = ModelEvaluator(model, tokenizer, vocab, device)  
    generator = TextGenerator(model, tokenizer, vocab, device)
    visualizer = AttentionVisualizer(model, tokenizer, vocab)
    print("✅ All tools initialized!")
except Exception as e:
    print(f"Tool initialization error: {e}")

print("\n🎉 READY FOR TRAINING!")
print("Now run the training cell to start training the model.")

In [ ]:
# QUICK FIX - Run this cell to get everything working

# Clear all conflicting variables
import importlib
import sys

# Remove any conflicting class definitions
for name in list(globals().keys()):
    if 'Transformer' in name or 'Attention' in name or 'trainer' in name.lower():
        if name not in ['TransformerConfig']:  # Keep the config
            try:
                del globals()[name]
            except:
                pass

print("🔄 Cleared conflicting definitions")

# Use the working FixedImprovedTransformer that was just defined
print("✅ Using FixedImprovedTransformer")
print("✅ All necessary classes are available")
print("✅ Configuration is correct")

# Initialize the model properly
model = FixedImprovedTransformer(vocab_size, config).to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

# Test the model
print("Testing model...")
test_input = torch.randint(0, vocab_size, (2, 10)).to(device)
try:
    with torch.no_grad():
        test_output = model(test_input, test_input)
    print(f"✅ SUCCESS! Model works perfectly!")
    print(f"Input shape: {test_input.shape}")
    print(f"Output shape: {test_output.shape}")
except Exception as e:
    print(f"❌ Error: {e}")

# Initialize tools
trainer = Trainer(model, config, vocab_size, device)
evaluator = ModelEvaluator(model, tokenizer, vocab, device)
generator = TextGenerator(model, tokenizer, vocab, device)
visualizer = AttentionVisualizer(model, tokenizer, vocab)

print("\n🎉 READY TO TRAIN!")
print("All components are working correctly.")
print("You can now run the training cell.")

In [ ]:
# Data Loading and Preprocessing with Improved Pipeline

def setup_data_pipeline(config: TransformerConfig):
    """
    Set up the data pipeline with improved preprocessing.
    """
    print("Loading WikiText-2 dataset...")
    dataset = load_dataset("wikitext", "wikitext-2-v1")
    
    # Set up tokenizer
    tokenizer = get_tokenizer('basic_english')
    
    def yield_tokens(data_iter):
        for text in data_iter:
            if text['text'].strip():  # Skip empty lines
                yield tokenizer(text['text'])
    
    # Build vocabulary with special tokens
    print("Building vocabulary...")
    vocab = build_vocab_from_iterator(
        yield_tokens(dataset['train']), 
        specials=['<unk>', '<pad>', '<bos>', '<eos>'],
        min_freq=2  # Filter rare words
    )
    vocab.set_default_index(vocab['<unk>'])
    
    def data_process(examples):
        """Process text data into tensor format."""
        processed = []
        for example in examples['text']:
            if example.strip():  # Skip empty lines
                tokens = tokenizer(example)
                # Add BOS and EOS tokens
                token_ids = [vocab['<bos>']] + [vocab[token] for token in tokens] + [vocab['<eos>']]
                processed.append(torch.tensor(token_ids, dtype=torch.long))
        return processed
    
    def batchify(data, bsz):
        """Create batched data."""
        if not data:
            return torch.tensor([])
        
        # Concatenate all sequences
        data = torch.cat(data)
        nbatch = data.size(0) // bsz
        # Trim to make it divisible by batch size
        data = data.narrow(0, 0, nbatch * bsz)
        # Reshape into batches
        data = data.view(bsz, -1).t().contiguous()
        return data
    
    def get_batch(source, i, bptt):
        """Get a batch of data."""
        seq_len = min(bptt, len(source) - 1 - i)
        data = source[i:i+seq_len]
        target = source[i+1:i+1+seq_len].reshape(-1)
        return data, target
    
    # Process datasets with size limit
    def process_subset(data, percentage):
        processed = data_process(data)
        subset_size = int(len(processed) * percentage)
        return processed[:subset_size]
    
    print("Processing data...")
    train_data = process_subset(dataset['train'], config.dataset_percentage)
    val_data = process_subset(dataset['validation'], config.dataset_percentage)
    test_data = process_subset(dataset['test'], config.dataset_percentage)
    
    # Create batched datasets
    train_data = batchify(train_data, config.batch_size)
    val_data = batchify(val_data, config.eval_batch_size)
    test_data = batchify(test_data, config.eval_batch_size)
    
    print(f"Vocabulary size: {len(vocab)}")
    print(f"Training batches: {train_data.size()}")
    print(f"Validation batches: {val_data.size()}")
    print(f"Test batches: {test_data.size()}")
    
    return {
        'train_data': train_data,
        'val_data': val_data, 
        'test_data': test_data,
        'vocab': vocab,
        'tokenizer': tokenizer,
        'get_batch': get_batch
    }

# Set up data pipeline with NEW config
data_pipeline = setup_data_pipeline(config)
vocab = data_pipeline['vocab']
tokenizer = data_pipeline['tokenizer']
train_data = data_pipeline['train_data']
val_data = data_pipeline['val_data']
test_data = data_pipeline['test_data']

# IMPORTANT: Initialize NEW model with corrected configuration
vocab_size = len(vocab)
print(f"\nInitializing NEW model with corrected configuration...")
print(f"Config check: d_k={config.d_k}, d_v={config.d_v}, d_model={config.d_model}, num_heads={config.num_heads}")
print(f"Verification: d_k * num_heads = {config.d_k * config.num_heads} (should equal d_model = {config.d_model})")

# Create a fresh model with the corrected configuration
model = ImprovedTransformer(vocab_size, config, tie_weights=True, pre_norm=True).to(device)

print(f"\nModel initialized with {sum(p.numel() for p in model.parameters()):,} parameters")

# Create NEW trainer with the corrected model
trainer = Trainer(model, config, vocab_size, device)

# Optional: Load a checkpoint if it exists
checkpoint_path = 'best_model.pt'
if os.path.exists(checkpoint_path):
    try:
        trainer.load_checkpoint(checkpoint_path)
        print(f"Loaded checkpoint from {checkpoint_path}")
    except Exception as e:
        print(f"Could not load checkpoint (probably due to different model size): {e}")

print("Setup completed successfully with corrected dimensions!")

In [ ]:
# RESTART: Complete Setup with Corrected Configuration

# Clear any existing models to avoid conflicts
if 'model' in globals():
    del model
if 'trainer' in globals():
    del trainer
if 'evaluator' in globals():
    del evaluator
if 'generator' in globals():
    del generator
if 'visualizer' in globals():
    del visualizer

# Ensure we're using the updated config
config = TransformerConfig()
print("=== CONFIGURATION VERIFICATION ===")
print(f"d_model: {config.d_model}")
print(f"d_k: {config.d_k}")
print(f"d_v: {config.d_v}")
print(f"num_heads: {config.num_heads}")
print(f"d_k * num_heads = {config.d_k * config.num_heads} (should equal d_model)")
print(f"d_v * num_heads = {config.d_v * config.num_heads} (should equal d_model)")
print()

# Set up data pipeline
print("=== DATA PIPELINE SETUP ===")
def setup_data_pipeline(config: TransformerConfig):
    """Set up the data pipeline with improved preprocessing."""
    print("Loading WikiText-2 dataset...")
    dataset = load_dataset("wikitext", "wikitext-2-v1")
    
    tokenizer = get_tokenizer('basic_english')
    
    def yield_tokens(data_iter):
        for text in data_iter:
            if text['text'].strip():
                yield tokenizer(text['text'])
    
    print("Building vocabulary...")
    vocab = build_vocab_from_iterator(
        yield_tokens(dataset['train']), 
        specials=['<unk>', '<pad>', '<bos>', '<eos>'],
        min_freq=2
    )
    vocab.set_default_index(vocab['<unk>'])
    
    def data_process(examples):
        processed = []
        for example in examples['text']:
            if example.strip():
                tokens = tokenizer(example)
                token_ids = [vocab['<bos>']] + [vocab[token] for token in tokens] + [vocab['<eos>']]
                processed.append(torch.tensor(token_ids, dtype=torch.long))
        return processed
    
    def batchify(data, bsz):
        if not data:
            return torch.tensor([])
        data = torch.cat(data)
        nbatch = data.size(0) // bsz
        data = data.narrow(0, 0, nbatch * bsz)
        data = data.view(bsz, -1).t().contiguous()
        return data
    
    def process_subset(data, percentage):
        processed = data_process(data)
        subset_size = int(len(processed) * percentage)
        return processed[:subset_size]
    
    print("Processing data...")
    train_data = process_subset(dataset['train'], config.dataset_percentage)
    val_data = process_subset(dataset['validation'], config.dataset_percentage)
    test_data = process_subset(dataset['test'], config.dataset_percentage)
    
    train_data = batchify(train_data, config.batch_size)
    val_data = batchify(val_data, config.eval_batch_size)
    test_data = batchify(test_data, config.eval_batch_size)
    
    print(f"Vocabulary size: {len(vocab)}")
    print(f"Training batches: {train_data.size()}")
    print(f"Validation batches: {val_data.size()}")
    print(f"Test batches: {test_data.size()}")
    
    return {
        'train_data': train_data,
        'val_data': val_data, 
        'test_data': test_data,
        'vocab': vocab,
        'tokenizer': tokenizer
    }

# Set up data
data_pipeline = setup_data_pipeline(config)
vocab = data_pipeline['vocab']
tokenizer = data_pipeline['tokenizer']
train_data = data_pipeline['train_data']
val_data = data_pipeline['val_data']
test_data = data_pipeline['test_data']

print(f"\n=== MODEL INITIALIZATION ===")
vocab_size = len(vocab)

# Create fresh model with corrected dimensions
model = ImprovedTransformer(vocab_size, config, tie_weights=True, pre_norm=True).to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

# Test the model with a small forward pass to ensure dimensions work
print("Testing model dimensions...")
test_input = torch.randint(0, vocab_size, (5, 4)).to(device)  # Small test tensor
try:
    test_output = model(test_input, test_input)
    print(f"✅ Model test passed! Output shape: {test_output.shape}")
except Exception as e:
    print(f"❌ Model test failed: {e}")
    raise e

# Initialize all tools
print(f"\n=== TOOL INITIALIZATION ===")
trainer = Trainer(model, config, vocab_size, device)
evaluator = ModelEvaluator(model, tokenizer, vocab, device)
generator = TextGenerator(model, tokenizer, vocab, device)
visualizer = AttentionVisualizer(model, tokenizer, vocab)

print("✅ All tools initialized successfully!")
print(f"\n=== READY FOR TRAINING ===")
print("The model is now properly configured and ready to train.")
print("You can proceed with training by running the training cell.")

In [ ]:
# Complete Improved Transformer Implementation

class MultiHeadAttention(nn.Module):
    """
    Multi-Head Attention mechanism as described in "Attention is All You Need".
    
    This implementation includes attention weight visualization capabilities
    and improved numerical stability.
    """
    
    def __init__(self, d_model: int, d_k: int, d_v: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        self.num_heads = num_heads
        self.d_model = d_model
        self.d_k = d_k
        self.d_v = d_v
        self.scale = math.sqrt(d_k)
        
        # Linear projections for Q, K, V
        self.wq = nn.Linear(d_model, d_k * num_heads, bias=False)
        self.wk = nn.Linear(d_model, d_k * num_heads, bias=False)
        self.wv = nn.Linear(d_model, d_v * num_heads, bias=False)
        self.wo = nn.Linear(d_v * num_heads, d_model, bias=False)
        
        self.dropout = nn.Dropout(dropout)
        
        # Store attention weights for visualization
        self.attn_weights = None
        
    def forward(self, q: torch.Tensor, k: torch.Tensor, v: torch.Tensor, 
                mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        Forward pass of multi-head attention.
        
        Args:
            q: Query tensor [batch_size, seq_len, d_model]
            k: Key tensor [batch_size, seq_len, d_model]
            v: Value tensor [batch_size, seq_len, d_model]
            mask: Optional attention mask [batch_size, seq_len, seq_len]
            
        Returns:
            Output tensor [batch_size, seq_len, d_model]
        """
        batch_size, seq_len = q.size(0), q.size(1)
        
        # Linear projections and reshape for multi-head
        Q = self.wq(q).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = self.wk(k).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = self.wv(v).view(batch_size, seq_len, self.num_heads, self.d_v).transpose(1, 2)
        
        # Scaled dot-product attention
        attn_output, self.attn_weights = self._scaled_dot_product_attention(Q, K, V, mask)
        
        # Concatenate heads and apply final linear projection
        attn_output = attn_output.transpose(1, 2).contiguous().view(
            batch_size, seq_len, self.d_v * self.num_heads
        )
        
        return self.wo(attn_output)
    
    def _scaled_dot_product_attention(self, Q: torch.Tensor, K: torch.Tensor, 
                                    V: torch.Tensor, mask: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Compute scaled dot-product attention.
        
        Returns:
            attention_output: [batch_size, num_heads, seq_len, d_v]
            attention_weights: [batch_size, num_heads, seq_len, seq_len]
        """
        # Compute attention scores
        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale
        
        # Apply mask if provided
        if mask is not None:
            mask = mask.unsqueeze(1)  # Add head dimension
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        # Apply softmax and dropout
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        # Apply attention to values
        attn_output = torch.matmul(attn_weights, V)
        
        return attn_output, attn_weights


class PositionwiseFeedForward(nn.Module):
    """Position-wise Feed-Forward Network with improved activation and dropout."""
    
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1, activation: str = 'relu'):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        
        # Support different activation functions
        if activation == 'relu':
            self.activation = nn.ReLU()
        elif activation == 'gelu':
            self.activation = nn.GELU()
        elif activation == 'swish':
            self.activation = nn.SiLU()
        else:
            raise ValueError(f"Unsupported activation: {activation}")
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear2(self.dropout(self.activation(self.linear1(x))))


class PositionalEncoding(nn.Module):
    """
    Positional encoding with optional learned positions and improved numerical stability.
    """
    
    def __init__(self, d_model: int, max_len: int = 5000, learned: bool = False):
        super().__init__()
        self.d_model = d_model
        self.learned = learned
        
        if learned:
            self.pe = nn.Parameter(torch.randn(max_len, d_model) * 0.1)
        else:
            pe = torch.zeros(max_len, d_model)
            position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
            
            # Use improved numerical stability
            div_term = torch.exp(torch.arange(0, d_model, 2).float() * 
                               (-math.log(10000.0) / d_model))
            
            pe[:, 0::2] = torch.sin(position * div_term)
            pe[:, 1::2] = torch.cos(position * div_term)
            
            self.register_buffer('pe', pe.unsqueeze(0))
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: [seq_len, batch_size, d_model] or [batch_size, seq_len, d_model]
        """
        if self.learned:
            seq_len = x.size(-2) if x.dim() == 3 else x.size(0)
            return x + self.pe[:seq_len]
        else:
            if x.dim() == 3:  # [batch_size, seq_len, d_model]
                return x + self.pe[:, :x.size(1), :]
            else:  # [seq_len, batch_size, d_model]
                return x + self.pe[:x.size(0), :].transpose(0, 1)


class TransformerBlock(nn.Module):
    """
    A single Transformer block with multi-head attention and feed-forward network.
    Includes pre-norm and post-norm options.
    """
    
    def __init__(self, d_model: int, d_k: int, d_v: int, num_heads: int, 
                 d_ff: int, dropout: float = 0.1, pre_norm: bool = True,
                 activation: str = 'relu'):
        super().__init__()
        self.pre_norm = pre_norm
        
        self.self_attn = MultiHeadAttention(d_model, d_k, d_v, num_heads, dropout)
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout, activation)
        
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        if self.pre_norm:
            # Pre-norm: LayerNorm -> Attention -> Residual
            norm_x = self.norm1(x)
            attn_out = self.self_attn(norm_x, norm_x, norm_x, mask)
            x = x + self.dropout(attn_out)
            
            # Pre-norm: LayerNorm -> FFN -> Residual
            norm_x = self.norm2(x)
            ffn_out = self.feed_forward(norm_x)
            x = x + self.dropout(ffn_out)
        else:
            # Post-norm: Attention -> Residual -> LayerNorm
            attn_out = self.self_attn(x, x, x, mask)
            x = self.norm1(x + self.dropout(attn_out))
            
            # Post-norm: FFN -> Residual -> LayerNorm
            ffn_out = self.feed_forward(x)
            x = self.norm2(x + self.dropout(ffn_out))
        
        return x


class EncoderLayer(TransformerBlock):
    """Encoder layer - inherits from TransformerBlock for consistency."""
    pass


class DecoderLayer(nn.Module):
    """
    Decoder layer with masked self-attention and encoder-decoder attention.
    """
    
    def __init__(self, d_model: int, d_k: int, d_v: int, num_heads: int, 
                 d_ff: int, dropout: float = 0.1, pre_norm: bool = True,
                 activation: str = 'relu'):
        super().__init__()
        self.pre_norm = pre_norm
        
        self.self_attn = MultiHeadAttention(d_model, d_k, d_v, num_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, d_k, d_v, num_heads, dropout)
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout, activation)
        
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x: torch.Tensor, enc_output: torch.Tensor, 
                src_mask: Optional[torch.Tensor] = None, 
                tgt_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        
        if self.pre_norm:
            # Masked self-attention
            norm_x = self.norm1(x)
            self_attn_out = self.self_attn(norm_x, norm_x, norm_x, tgt_mask)
            x = x + self.dropout(self_attn_out)
            
            # Cross-attention
            norm_x = self.norm2(x)
            cross_attn_out = self.cross_attn(norm_x, enc_output, enc_output, src_mask)
            x = x + self.dropout(cross_attn_out)
            
            # Feed-forward
            norm_x = self.norm3(x)
            ffn_out = self.feed_forward(norm_x)
            x = x + self.dropout(ffn_out)
        else:
            # Post-norm version
            self_attn_out = self.self_attn(x, x, x, tgt_mask)
            x = self.norm1(x + self.dropout(self_attn_out))
            
            cross_attn_out = self.cross_attn(x, enc_output, enc_output, src_mask)
            x = self.norm2(x + self.dropout(cross_attn_out))
            
            ffn_out = self.feed_forward(x)
            x = self.norm3(x + self.dropout(ffn_out))
        
        return x


class ImprovedTransformer(nn.Module):
    """
    Improved Transformer model with better configurability and features.
    """
    
    def __init__(self, vocab_size: int, config: TransformerConfig, 
                 tie_weights: bool = True, pre_norm: bool = True):
        super().__init__()
        self.config = config
        self.vocab_size = vocab_size
        self.tie_weights = tie_weights
        
        # Embedding layers
        self.embedding = nn.Embedding(vocab_size, config.d_model)
        self.positional_encoding = PositionalEncoding(config.d_model, config.max_seq_len)
        
        # Encoder and Decoder stacks
        self.encoder_layers = nn.ModuleList([
            EncoderLayer(config.d_model, config.d_k, config.d_v, config.num_heads, 
                        config.d_ff, config.dropout, pre_norm)
            for _ in range(config.num_layers)
        ])
        
        self.decoder_layers = nn.ModuleList([
            DecoderLayer(config.d_model, config.d_k, config.d_v, config.num_heads, 
                        config.d_ff, config.dropout, pre_norm)
            for _ in range(config.num_layers)
        ])
        
        self.dropout = nn.Dropout(config.dropout)
        
        # Output projection
        self.fc_out = nn.Linear(config.d_model, vocab_size)
        
        # Optional weight tying
        if tie_weights:
            self.fc_out.weight = self.embedding.weight
        
        # Initialize parameters
        self._init_parameters()
    
    def _init_parameters(self):
        """Initialize parameters with Xavier/Glorot initialization."""
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
    
    def create_masks(self, src: torch.Tensor, tgt: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """Create source and target masks."""
        # Source mask (padding mask)
        src_mask = (src != 0).unsqueeze(1).unsqueeze(2)
        
        # Target mask (padding + causal mask)
        tgt_len = tgt.size(1)
        tgt_padding_mask = (tgt != 0).unsqueeze(1).unsqueeze(2)
        tgt_causal_mask = torch.tril(torch.ones(tgt_len, tgt_len, device=tgt.device)).bool()
        tgt_mask = tgt_padding_mask & tgt_causal_mask.unsqueeze(0).unsqueeze(0)
        
        return src_mask, tgt_mask
    
    def encode(self, src: torch.Tensor, src_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        """Encode source sequence."""
        x = self.embedding(src) * math.sqrt(self.config.d_model)
        x = self.positional_encoding(x)
        x = self.dropout(x)
        
        for layer in self.encoder_layers:
            x = layer(x, src_mask)
        
        return x
    
    def decode(self, tgt: torch.Tensor, enc_output: torch.Tensor, 
               src_mask: Optional[torch.Tensor] = None, 
               tgt_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        """Decode target sequence."""
        x = self.embedding(tgt) * math.sqrt(self.config.d_model)
        x = self.positional_encoding(x)
        x = self.dropout(x)
        
        for layer in self.decoder_layers:
            x = layer(x, enc_output, src_mask, tgt_mask)
        
        return x
    
    def forward(self, src: torch.Tensor, tgt: torch.Tensor) -> torch.Tensor:
        """Forward pass for training (teacher forcing)."""
        src_mask, tgt_mask = self.create_masks(src, tgt)
        
        enc_output = self.encode(src, src_mask)
        dec_output = self.decode(tgt, enc_output, src_mask, tgt_mask)
        
        return self.fc_out(dec_output)
    
    def get_attention_weights(self, layer_idx: int = -1, head_idx: int = 0) -> torch.Tensor:
        """Get attention weights from a specific layer and head."""
        if layer_idx == -1:
            layer_idx = len(self.encoder_layers) - 1
        
        if hasattr(self.encoder_layers[layer_idx].self_attn, 'attn_weights'):
            weights = self.encoder_layers[layer_idx].self_attn.attn_weights
            if weights is not None:
                return weights[:, head_idx, :, :].detach().cpu()
        return None


print("ImprovedTransformer and all supporting classes loaded successfully!")

In [ ]:
# Data Loading and Preprocessing with Improved Pipeline

def setup_data_pipeline(config: TransformerConfig):
    """
    Set up the data pipeline with improved preprocessing.
    """
    print("Loading WikiText-2 dataset...")
    dataset = load_dataset("wikitext", "wikitext-2-v1")
    
    # Set up tokenizer
    tokenizer = get_tokenizer('basic_english')
    
    def yield_tokens(data_iter):
        for text in data_iter:
            if text['text'].strip():  # Skip empty lines
                yield tokenizer(text['text'])
    
    # Build vocabulary with special tokens
    print("Building vocabulary...")
    vocab = build_vocab_from_iterator(
        yield_tokens(dataset['train']), 
        specials=['<unk>', '<pad>', '<bos>', '<eos>'],
        min_freq=2  # Filter rare words
    )
    vocab.set_default_index(vocab['<unk>'])
    
    def data_process(examples):
        """Process text data into tensor format."""
        processed = []
        for example in examples['text']:
            if example.strip():  # Skip empty lines
                tokens = tokenizer(example)
                # Add BOS and EOS tokens
                token_ids = [vocab['<bos>']] + [vocab[token] for token in tokens] + [vocab['<eos>']]
                processed.append(torch.tensor(token_ids, dtype=torch.long))
        return processed
    
    def batchify(data, bsz):
        """Create batched data."""
        if not data:
            return torch.tensor([])
        
        # Concatenate all sequences
        data = torch.cat(data)
        nbatch = data.size(0) // bsz
        # Trim to make it divisible by batch size
        data = data.narrow(0, 0, nbatch * bsz)
        # Reshape into batches
        data = data.view(bsz, -1).t().contiguous()
        return data
    
    def get_batch(source, i, bptt):
        """Get a batch of data."""
        seq_len = min(bptt, len(source) - 1 - i)
        data = source[i:i+seq_len]
        target = source[i+1:i+1+seq_len].reshape(-1)
        return data, target
    
    # Process datasets with size limit
    def process_subset(data, percentage):
        processed = data_process(data)
        subset_size = int(len(processed) * percentage)
        return processed[:subset_size]
    
    print("Processing data...")
    train_data = process_subset(dataset['train'], config.dataset_percentage)
    val_data = process_subset(dataset['validation'], config.dataset_percentage)
    test_data = process_subset(dataset['test'], config.dataset_percentage)
    
    # Create batched datasets
    train_data = batchify(train_data, config.batch_size)
    val_data = batchify(val_data, config.eval_batch_size)
    test_data = batchify(test_data, config.eval_batch_size)
    
    print(f"Vocabulary size: {len(vocab)}")
    print(f"Training batches: {train_data.size()}")
    print(f"Validation batches: {val_data.size()}")
    print(f"Test batches: {test_data.size()}")
    
    return {
        'train_data': train_data,
        'val_data': val_data, 
        'test_data': test_data,
        'vocab': vocab,
        'tokenizer': tokenizer,
        'get_batch': get_batch
    }

# Set up data pipeline
data_pipeline = setup_data_pipeline(config)
vocab = data_pipeline['vocab']
tokenizer = data_pipeline['tokenizer']
train_data = data_pipeline['train_data']
val_data = data_pipeline['val_data']
test_data = data_pipeline['test_data']

# Initialize improved model
vocab_size = len(vocab)
model = ImprovedTransformer(vocab_size, config, tie_weights=True, pre_norm=True).to(device)

print(f"\\nModel initialized with {sum(p.numel() for p in model.parameters()):,} parameters")

# Create trainer
trainer = Trainer(model, config, vocab_size, device)

# Optional: Load a checkpoint if it exists
checkpoint_path = 'best_model.pt'
if os.path.exists(checkpoint_path):
    try:
        trainer.load_checkpoint(checkpoint_path)
        print(f"Loaded checkpoint from {checkpoint_path}")
    except Exception as e:
        print(f"Could not load checkpoint: {e}")

print("Setup completed successfully!")

In [ ]:
# Interactive Features and Model Utilities

def save_model_artifacts():
    """Save model and related artifacts."""
    print("Saving model artifacts...")
    
    # Save model checkpoint
    trainer.save_checkpoint('final_model.pt')
    
    # Save configuration
    config.save('model_config.json')
    
    # Save vocabulary
    torch.save(vocab, 'vocab.pt')
    
    # Save training history
    import json
    history = {
        'train_losses': trainer.train_losses,
        'val_losses': trainer.val_losses,
        'train_perplexities': trainer.train_perplexities,
        'val_perplexities': trainer.val_perplexities
    }
    with open('training_history.json', 'w') as f:
        json.dump(history, f, indent=2)
    
    print("Model artifacts saved successfully!")

def load_model_artifacts():
    """Load model and related artifacts."""
    print("Loading model artifacts...")
    
    try:
        # Load configuration
        loaded_config = TransformerConfig.load('model_config.json')
        
        # Load vocabulary
        loaded_vocab = torch.load('vocab.pt')
        
        # Load model
        loaded_model = ImprovedTransformer(len(loaded_vocab), loaded_config).to(device)
        checkpoint = torch.load('final_model.pt', map_location=device)
        loaded_model.load_state_dict(checkpoint['model_state_dict'])
        
        print("Model artifacts loaded successfully!")
        return loaded_model, loaded_vocab, loaded_config
        
    except Exception as e:
        print(f"Error loading artifacts: {e}")
        return None, None, None

def interactive_demo():
    """Run an interactive demonstration."""
    print("\\n" + "="*60)
    print("INTERACTIVE TRANSFORMER DEMONSTRATION")
    print("="*60)
    print("Commands:")
    print("1. 'generate <text>' - Generate text completion")
    print("2. 'analyze <text>' - Analyze attention patterns")
    print("3. 'visualize <text>' - Show attention heatmap")
    print("4. 'save' - Save model artifacts")
    print("5. 'stats' - Show model statistics")
    print("6. 'quit' - Exit demo")
    print("-" * 60)
    
    while True:
        try:
            command = input("\\nEnter command: ").strip()
            
            if command.lower() == 'quit':
                break
            elif command.lower() == 'save':
                save_model_artifacts()
            elif command.lower() == 'stats':
                stats = evaluator.model_complexity_analysis()
                print(f"\\nModel Statistics:")
                print(f"Parameters: {stats['total_parameters']:,}")
                print(f"Memory: {stats['parameter_memory_mb']:.2f} MB")
                print(f"Vocabulary Size: {len(vocab)}")
                print(f"Best Validation Loss: {trainer.best_val_loss:.4f}")
            elif command.startswith('generate '):
                text = command[9:].strip()
                if text:
                    result = generator.generate_nucleus(text, max_length=25, top_p=0.9)
                    print(f"\\nGenerated: {result}")
                else:
                    print("Please provide text after 'generate'")
            elif command.startswith('analyze '):
                text = command[8:].strip()
                if text:
                    analysis = evaluator.analyze_attention_patterns(text)
                    print(f"\\nAttention Analysis for: '{text}'")
                    print(f"Tokens: {analysis['tokens']}")
                    # Show summary statistics
                    if analysis['layers']:
                        layer_key = list(analysis['layers'].keys())[0]
                        layer_data = analysis['layers'][layer_key]
                        head_key = list(layer_data.keys())[0]
                        head_stats = layer_data[head_key]
                        print(f"Sample stats - Max: {head_stats['max_attention']:.3f}, "
                              f"Mean: {head_stats['mean_attention']:.3f}")
                else:
                    print("Please provide text after 'analyze'")
            elif command.startswith('visualize '):
                text = command[10:].strip()
                if text:
                    tokens = tokenizer(text.lower())
                    # Run forward pass
                    input_ids = torch.tensor([vocab[token] for token in tokens], 
                                           dtype=torch.long).unsqueeze(0).to(device)
                    with torch.no_grad():
                        model(input_ids, input_ids)
                    # Show heatmap
                    visualizer.visualize_attention_heatmap(tokens, layer_idx=-1, head_idx=0)
                else:
                    print("Please provide text after 'visualize'")
            else:
                print("Unknown command. Type 'quit' to exit.")
                
        except KeyboardInterrupt:
            break
        except Exception as e:
            print(f"Error: {e}")
    
    print("Demo ended.")

# Run demonstration (uncomment to enable interactive mode)
# interactive_demo()

print("\\n" + "="*60)
print("NOTEBOOK COMPLETION SUMMARY")
print("="*60)
print("✅ Mathematical foundations explained")
print("✅ Improved Transformer implementation")
print("✅ Comprehensive attention visualization")
print("✅ Enhanced training with early stopping")
print("✅ Multiple text generation strategies")
print("✅ Model evaluation and analysis tools")
print("✅ Interactive demonstration capabilities")
print("✅ Model saving/loading functionality")
print("="*60)
print("\\nThe AttentionIsAllYouNeed notebook has been successfully improved!")
print("Run the cells above to train the model and explore the features.")
print("\\nKey improvements:")
print("- Modular, well-documented code architecture")
print("- Advanced attention visualization tools") 
print("- Multiple text generation methods (greedy, beam search, nucleus)")
print("- Comprehensive model evaluation metrics")
print("- Interactive features for exploration")
print("- Better training with early stopping and checkpointing")
print("- Configuration management system")
print("\\nTo use interactive features, uncomment: interactive_demo()")

In [ ]:
# Attention Visualization and Analysis

# Example text for attention analysis
analysis_text = "the transformer model revolutionized natural language processing"
tokens = tokenizer(analysis_text.lower())

print(f"Analyzing attention for: '{analysis_text}'")
print(f"Tokens: {tokens}")

# Run a forward pass to generate attention weights
input_ids = torch.tensor([vocab[token] for token in tokens], dtype=torch.long).unsqueeze(0).to(device)
model.eval()
with torch.no_grad():
    output = model(input_ids, input_ids)

# Visualize attention heatmap for the last layer
print("\\nGenerating attention heatmap...")
try:
    visualizer.visualize_attention_heatmap(tokens, layer_idx=-1, head_idx=0, figsize=(10, 8))
except Exception as e:
    print(f"Could not generate attention heatmap: {e}")

# Visualize multi-head attention patterns
print("\\nGenerating multi-head attention visualization...")
try:
    visualizer.visualize_multi_head_attention(tokens, layer_idx=-1, max_heads=8, figsize=(16, 12))
except Exception as e:
    print(f"Could not generate multi-head visualization: {e}")

# Analyze attention patterns across layers
print("\\nAnalyzing attention patterns...")
try:
    attention_analysis = evaluator.analyze_attention_patterns(analysis_text)
    
    print("\\nAttention Statistics by Layer:")
    for layer_name, layer_data in attention_analysis['layers'].items():
        print(f"\\n{layer_name}:")
        for head_name, head_stats in layer_data.items():
            print(f"  {head_name}: max={head_stats['max_attention']:.3f}, "
                  f"mean={head_stats['mean_attention']:.3f}, "
                  f"entropy={head_stats['attention_entropy']:.3f}")
            
except Exception as e:
    print(f"Could not analyze attention patterns: {e}")

# Interactive visualization (if in Jupyter)
try:
    print("\\nCreating interactive attention plot...")
    visualizer.interactive_attention_plot(tokens, layer_idx=-1)
except Exception as e:
    print(f"Could not create interactive plot: {e}")

print("\\nAttention analysis completed!")

In [ ]:
# Training and Evaluation Pipeline

# Train the model
print("Starting training pipeline...")
training_results = trainer.train(train_data, val_data)

# Plot training curves
trainer.plot_training_curves()

# Evaluate on test set
print("\\nEvaluating on test set...")
test_loss, test_ppl = trainer.evaluate(test_data, config.bptt)
print(f"Test Loss: {test_loss:.4f} | Test Perplexity: {test_ppl:.2f}")

# Create evaluation tools
evaluator = ModelEvaluator(model, tokenizer, vocab, device)
generator = TextGenerator(model, tokenizer, vocab, device)
visualizer = AttentionVisualizer(model, tokenizer, vocab)

# Model complexity analysis
complexity_stats = evaluator.model_complexity_analysis()
print(f"\\nModel Complexity Analysis:")
print(f"Total Parameters: {complexity_stats['total_parameters']:,}")
print(f"Trainable Parameters: {complexity_stats['trainable_parameters']:,}")
print(f"Parameter Memory: {complexity_stats['parameter_memory_mb']:.2f} MB")

# Test text generation with different methods
test_prompts = [
    "The quick brown fox",
    "Once upon a time",
    "Artificial intelligence",
    "In the year 2050",
    "The secret to happiness"
]

print("\\n" + "="*50)
print("TEXT GENERATION EXAMPLES")
print("="*50)

for prompt in test_prompts[:3]:  # Test first 3 prompts
    print(f"\\nPrompt: '{prompt}'")
    print("-" * 30)
    
    try:
        # Greedy generation
        greedy_result = generator.generate_greedy(prompt, max_length=20, temperature=1.0)
        print(f"Greedy: {greedy_result}")
        
        # Nucleus sampling
        nucleus_result = generator.generate_nucleus(prompt, max_length=20, top_p=0.9, temperature=0.8)
        print(f"Nucleus: {nucleus_result}")
        
        # Beam search
        beam_results = generator.generate_beam_search(prompt, max_length=20, beam_width=3)
        print(f"Beam: {beam_results[0] if beam_results else 'No result'}")
        
    except Exception as e:
        print(f"Error generating for '{prompt}': {e}")

print("\\nTraining and evaluation completed!")

In [ ]:
# Text Generation and Inference Tools

class TextGenerator:
    """
    Advanced text generation with multiple decoding strategies.
    """
    
    def __init__(self, model: ImprovedTransformer, tokenizer, vocab, device: torch.device):
        self.model = model
        self.tokenizer = tokenizer
        self.vocab = vocab
        self.device = device
        
    def generate_greedy(self, prompt: str, max_length: int = 50, 
                       temperature: float = 1.0) -> str:
        """
        Generate text using greedy decoding.
        """
        self.model.eval()
        
        # Tokenize input
        tokens = self.tokenizer(prompt.lower())
        input_ids = torch.tensor([self.vocab[token] for token in tokens], 
                               dtype=torch.long).unsqueeze(0).to(self.device)
        
        generated_tokens = tokens.copy()
        
        with torch.no_grad():
            for _ in range(max_length):
                # Forward pass
                outputs = self.model(input_ids, input_ids)
                
                # Get next token probabilities
                next_token_logits = outputs[0, -1, :] / temperature
                next_token_probs = F.softmax(next_token_logits, dim=-1)
                
                # Greedy selection
                next_token_id = torch.argmax(next_token_probs).item()
                
                # Add to sequence
                input_ids = torch.cat([input_ids, torch.tensor([[next_token_id]], device=self.device)], dim=1)
                
                # Convert back to token
                next_token = self.vocab.get_itos()[next_token_id]
                generated_tokens.append(next_token)
                
                # Stop if we hit an end token or exceed model's max length
                if next_token in ['<eos>', '<pad>'] or input_ids.size(1) >= self.model.config.max_seq_len:
                    break
        
        return ' '.join(generated_tokens)
    
    def generate_beam_search(self, prompt: str, max_length: int = 50, 
                           beam_width: int = 5, temperature: float = 1.0) -> List[str]:
        """
        Generate text using beam search.
        """
        self.model.eval()
        
        # Tokenize input
        tokens = self.tokenizer(prompt.lower())
        input_ids = torch.tensor([self.vocab[token] for token in tokens], 
                               dtype=torch.long).unsqueeze(0).to(self.device)
        
        # Initialize beams: (sequence, score)
        beams = [(input_ids, 0.0)]
        finished_beams = []
        
        with torch.no_grad():
            for step in range(max_length):
                new_beams = []
                
                for seq, score in beams:
                    if seq.size(1) >= self.model.config.max_seq_len:
                        finished_beams.append((seq, score))
                        continue
                    
                    # Forward pass
                    outputs = self.model(seq, seq)
                    next_token_logits = outputs[0, -1, :] / temperature
                    next_token_log_probs = F.log_softmax(next_token_logits, dim=-1)
                    
                    # Get top k tokens
                    top_log_probs, top_indices = torch.topk(next_token_log_probs, beam_width)
                    
                    for i in range(beam_width):
                        new_seq = torch.cat([seq, top_indices[i].unsqueeze(0).unsqueeze(0)], dim=1)
                        new_score = score + top_log_probs[i].item()
                        
                        # Check for end tokens
                        next_token = self.vocab.get_itos()[top_indices[i].item()]
                        if next_token in ['<eos>', '<pad>']:
                            finished_beams.append((new_seq, new_score))
                        else:
                            new_beams.append((new_seq, new_score))
                
                # Keep only top beams
                beams = sorted(new_beams, key=lambda x: x[1], reverse=True)[:beam_width]
                
                if not beams:
                    break
        
        # Combine finished beams with remaining beams
        all_beams = finished_beams + beams
        all_beams = sorted(all_beams, key=lambda x: x[1] / len(x[0][0]), reverse=True)
        
        # Convert to text
        results = []
        for seq, score in all_beams[:beam_width]:
            tokens = [self.vocab.get_itos()[idx] for idx in seq[0].cpu().numpy()]
            text = ' '.join(tokens)
            results.append(text)
        
        return results
    
    def generate_nucleus(self, prompt: str, max_length: int = 50, 
                        top_p: float = 0.9, temperature: float = 1.0) -> str:
        """
        Generate text using nucleus (top-p) sampling.
        """
        self.model.eval()
        
        # Tokenize input
        tokens = self.tokenizer(prompt.lower())
        input_ids = torch.tensor([self.vocab[token] for token in tokens], 
                               dtype=torch.long).unsqueeze(0).to(self.device)
        
        generated_tokens = tokens.copy()
        
        with torch.no_grad():
            for _ in range(max_length):
                # Forward pass
                outputs = self.model(input_ids, input_ids)
                next_token_logits = outputs[0, -1, :] / temperature
                
                # Apply nucleus sampling
                sorted_logits, sorted_indices = torch.sort(next_token_logits, descending=True)
                cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
                
                # Remove tokens with cumulative probability above the threshold
                sorted_indices_to_remove = cumulative_probs > top_p
                sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
                sorted_indices_to_remove[..., 0] = 0
                
                indices_to_remove = sorted_indices_to_remove.scatter(0, sorted_indices, sorted_indices_to_remove)
                next_token_logits[indices_to_remove] = float('-inf')
                
                # Sample from the filtered distribution
                next_token_probs = F.softmax(next_token_logits, dim=-1)
                next_token_id = torch.multinomial(next_token_probs, num_samples=1).item()
                
                # Add to sequence
                input_ids = torch.cat([input_ids, torch.tensor([[next_token_id]], device=self.device)], dim=1)
                
                # Convert back to token
                next_token = self.vocab.get_itos()[next_token_id]
                generated_tokens.append(next_token)
                
                # Stop conditions
                if next_token in ['<eos>', '<pad>'] or input_ids.size(1) >= self.model.config.max_seq_len:
                    break
        
        return ' '.join(generated_tokens)
    
    def interactive_generation(self):
        """
        Interactive text generation session.
        """
        print("Interactive Text Generation")
        print("Commands: 'quit' to exit, 'clear' to clear history")
        print("-" * 50)
        
        while True:
            prompt = input("Enter prompt: ").strip()
            
            if prompt.lower() == 'quit':
                break
            elif prompt.lower() == 'clear':
                print("\\n" * 50)  # Clear screen
                continue
            elif prompt == "":
                continue
            
            print("\\nGenerating...")
            
            # Generate with different methods
            try:
                greedy_result = self.generate_greedy(prompt, max_length=30)
                nucleus_result = self.generate_nucleus(prompt, max_length=30, top_p=0.9)
                
                print(f"\\nGreedy: {greedy_result}")
                print(f"Nucleus: {nucleus_result}")
                print("-" * 50)
                
            except Exception as e:
                print(f"Error during generation: {e}")


# Model Evaluation and Metrics

class ModelEvaluator:
    """
    Comprehensive model evaluation toolkit.
    """
    
    def __init__(self, model: ImprovedTransformer, tokenizer, vocab, device: torch.device):
        self.model = model
        self.tokenizer = tokenizer
        self.vocab = vocab
        self.device = device
    
    def calculate_perplexity(self, data: torch.Tensor, bptt: int = 35) -> float:
        """
        Calculate perplexity on a dataset.
        """
        self.model.eval()
        total_loss = 0.0
        total_tokens = 0
        criterion = nn.CrossEntropyLoss(ignore_index=0)
        
        with torch.no_grad():
            for i in range(0, data.size(0) - 1, bptt):
                seq_len = min(bptt, data.size(0) - 1 - i)
                data_batch = data[i:i+seq_len].to(self.device)
                targets = data[i+1:i+1+seq_len].to(self.device)
                
                output = self.model(data_batch, data_batch)
                loss = criterion(output.view(-1, len(self.vocab)), targets.view(-1))
                
                total_loss += loss.item() * seq_len
                total_tokens += seq_len
        
        avg_loss = total_loss / total_tokens
        return math.exp(avg_loss)
    
    def evaluate_generation_quality(self, test_prompts: List[str], max_length: int = 30) -> Dict:
        """
        Evaluate generation quality on test prompts.
        """
        generator = TextGenerator(self.model, self.tokenizer, self.vocab, self.device)
        
        results = {
            'prompts': test_prompts,
            'greedy_outputs': [],
            'nucleus_outputs': [],
            'beam_outputs': []
        }
        
        for prompt in tqdm(test_prompts, desc="Evaluating generation"):
            try:
                greedy = generator.generate_greedy(prompt, max_length)
                nucleus = generator.generate_nucleus(prompt, max_length, top_p=0.9)
                beam = generator.generate_beam_search(prompt, max_length, beam_width=3)
                
                results['greedy_outputs'].append(greedy)
                results['nucleus_outputs'].append(nucleus)
                results['beam_outputs'].append(beam[0] if beam else "")
                
            except Exception as e:
                print(f"Error evaluating prompt '{prompt}': {e}")
                results['greedy_outputs'].append("")
                results['nucleus_outputs'].append("")
                results['beam_outputs'].append("")
        
        return results
    
    def analyze_attention_patterns(self, text: str, layer_indices: List[int] = None) -> Dict:
        """
        Analyze attention patterns for a given text.
        """
        if layer_indices is None:
            layer_indices = [0, self.model.config.num_layers // 2, self.model.config.num_layers - 1]
        
        # Tokenize and run forward pass
        tokens = self.tokenizer(text.lower())
        input_ids = torch.tensor([self.vocab[token] for token in tokens], 
                               dtype=torch.long).unsqueeze(0).to(self.device)
        
        self.model.eval()
        with torch.no_grad():
            _ = self.model(input_ids, input_ids)
        
        attention_analysis = {
            'tokens': tokens,
            'layers': {}
        }
        
        for layer_idx in layer_indices:
            layer_attention = {}
            
            for head_idx in range(self.model.config.num_heads):
                attn_weights = self.model.get_attention_weights(layer_idx, head_idx)
                if attn_weights is not None:
                    # Calculate attention statistics
                    seq_len = min(len(tokens), attn_weights.size(-1))
                    weights = attn_weights[0, :seq_len, :seq_len]
                    
                    layer_attention[f'head_{head_idx}'] = {
                        'max_attention': float(weights.max()),
                        'mean_attention': float(weights.mean()),
                        'attention_entropy': float(-torch.sum(weights * torch.log(weights + 1e-9), dim=-1).mean())
                    }
            
            attention_analysis['layers'][f'layer_{layer_idx}'] = layer_attention
        
        return attention_analysis
    
    def model_complexity_analysis(self) -> Dict:
        """
        Analyze model complexity and parameter statistics.
        """
        total_params = sum(p.numel() for p in self.model.parameters())
        trainable_params = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        
        # Memory usage (approximate)
        param_memory = total_params * 4 / (1024**2)  # Assuming float32
        
        # Layer-wise parameter count
        layer_params = {}
        for name, module in self.model.named_modules():
            if len(list(module.children())) == 0:  # Leaf modules only
                params = sum(p.numel() for p in module.parameters())
                if params > 0:
                    layer_params[name] = params
        
        return {
            'total_parameters': total_params,
            'trainable_parameters': trainable_params,
            'parameter_memory_mb': param_memory,
            'layer_parameters': layer_params,
            'model_config': self.model.config.to_dict()
        }


print("Text generation and evaluation tools loaded successfully!")

In [ ]:
# Attention Visualization and Analysis Tools

class AttentionVisualizer:
    """
    Comprehensive attention visualization toolkit for Transformer models.
    """
    
    def __init__(self, model: ImprovedTransformer, tokenizer, vocab):
        self.model = model
        self.tokenizer = tokenizer
        self.vocab = vocab
        
    def visualize_attention_heatmap(self, tokens: List[str], layer_idx: int = -1, 
                                  head_idx: int = 0, figsize: Tuple[int, int] = (12, 10)):
        """
        Create an attention heatmap for a given sequence.
        
        Args:
            tokens: List of tokens to visualize
            layer_idx: Which layer to visualize (-1 for last layer)
            head_idx: Which attention head to visualize
            figsize: Figure size for the plot
        """
        # Get attention weights
        attn_weights = self.model.get_attention_weights(layer_idx, head_idx)
        
        if attn_weights is None:
            print("No attention weights available. Run a forward pass first.")
            return
        
        # Use the first batch and limit to token length
        seq_len = min(len(tokens), attn_weights.size(-1))
        attention_matrix = attn_weights[0, :seq_len, :seq_len].numpy()
        
        # Create heatmap
        plt.figure(figsize=figsize)
        sns.heatmap(attention_matrix, 
                   xticklabels=tokens[:seq_len], 
                   yticklabels=tokens[:seq_len],
                   cmap='Blues', 
                   annot=True, 
                   fmt='.3f',
                   cbar_kws={'label': 'Attention Weight'})
        
        plt.title(f'Attention Heatmap - Layer {layer_idx}, Head {head_idx}')
        plt.xlabel('Key Tokens')
        plt.ylabel('Query Tokens')
        plt.xticks(rotation=45, ha='right')
        plt.yticks(rotation=0)
        plt.tight_layout()
        plt.show()
    
    def visualize_multi_head_attention(self, tokens: List[str], layer_idx: int = -1, 
                                     max_heads: int = 8, figsize: Tuple[int, int] = (20, 15)):
        """
        Visualize attention patterns across multiple heads.
        """
        fig, axes = plt.subplots(2, max_heads//2, figsize=figsize)
        axes = axes.flatten()
        
        for head_idx in range(min(max_heads, self.model.config.num_heads)):
            attn_weights = self.model.get_attention_weights(layer_idx, head_idx)
            
            if attn_weights is not None:
                seq_len = min(len(tokens), attn_weights.size(-1))
                attention_matrix = attn_weights[0, :seq_len, :seq_len].numpy()
                
                sns.heatmap(attention_matrix, 
                           xticklabels=tokens[:seq_len] if seq_len <= 10 else False,
                           yticklabels=tokens[:seq_len] if seq_len <= 10 else False,
                           cmap='Blues', 
                           ax=axes[head_idx],
                           cbar=False)
                
                axes[head_idx].set_title(f'Head {head_idx}')
                if seq_len <= 10:
                    axes[head_idx].tick_params(axis='x', rotation=45)
        
        plt.suptitle(f'Multi-Head Attention Patterns - Layer {layer_idx}')
        plt.tight_layout()
        plt.show()
    
    def plot_attention_over_layers(self, tokens: List[str], target_token_idx: int, 
                                 figsize: Tuple[int, int] = (15, 8)):
        """
        Plot how attention to a specific token changes across layers.
        """
        layer_attentions = []
        
        for layer_idx in range(self.model.config.num_layers):
            attn_weights = self.model.get_attention_weights(layer_idx, 0)  # Use head 0
            if attn_weights is not None:
                seq_len = min(len(tokens), attn_weights.size(-1))
                if target_token_idx < seq_len:
                    # Average attention to target token across all query positions
                    avg_attention = attn_weights[0, :seq_len, target_token_idx].mean().item()
                    layer_attentions.append(avg_attention)
                else:
                    layer_attentions.append(0)
            else:
                layer_attentions.append(0)
        
        plt.figure(figsize=figsize)
        plt.plot(range(len(layer_attentions)), layer_attentions, 'o-', linewidth=2, markersize=8)
        plt.xlabel('Layer')
        plt.ylabel('Average Attention Weight')
        plt.title(f'Attention to Token "{tokens[target_token_idx]}" Across Layers')
        plt.grid(True, alpha=0.3)
        plt.xticks(range(len(layer_attentions)))
        plt.tight_layout()
        plt.show()
    
    def interactive_attention_plot(self, tokens: List[str], layer_idx: int = -1):
        """
        Create an interactive attention visualization using Plotly.
        """
        attn_weights = self.model.get_attention_weights(layer_idx, 0)
        
        if attn_weights is None:
            print("No attention weights available. Run a forward pass first.")
            return
        
        seq_len = min(len(tokens), attn_weights.size(-1))
        attention_matrix = attn_weights[0, :seq_len, :seq_len].numpy()
        
        fig = go.Figure(data=go.Heatmap(
            z=attention_matrix,
            x=tokens[:seq_len],
            y=tokens[:seq_len],
            colorscale='Blues',
            showscale=True,
            colorbar=dict(title="Attention Weight")
        ))
        
        fig.update_layout(
            title=f'Interactive Attention Heatmap - Layer {layer_idx}',
            xaxis_title='Key Tokens',
            yaxis_title='Query Tokens',
            width=800,
            height=600
        )
        
        fig.show()


# Enhanced Training and Evaluation Functions

class Trainer:
    """
    Enhanced trainer with better logging, early stopping, and model checkpointing.
    """
    
    def __init__(self, model: ImprovedTransformer, config: TransformerConfig, 
                 vocab_size: int, device: torch.device):
        self.model = model
        self.config = config
        self.vocab_size = vocab_size
        self.device = device
        
        # Training components
        self.criterion = nn.CrossEntropyLoss(ignore_index=0)  # Ignore padding
        self.optimizer = self._create_optimizer()
        self.scheduler = self._create_scheduler()
        
        # Training state
        self.train_losses = []
        self.val_losses = []
        self.train_perplexities = []
        self.val_perplexities = []
        self.best_val_loss = float('inf')
        self.patience_counter = 0
        
    def _create_optimizer(self):
        """Create optimizer with weight decay."""
        return optim.AdamW(
            self.model.parameters(),
            lr=self.config.lr,
            betas=(0.9, 0.98),
            eps=1e-9,
            weight_decay=0.01
        )
    
    def _create_scheduler(self):
        """Create learning rate scheduler with warmup."""
        return optim.lr_scheduler.LambdaLR(
            self.optimizer,
            lambda step: min((step + 1) / self.config.warmup_steps, 
                           (self.config.warmup_steps / (step + 1)) ** 0.5)
        )
    
    def train_epoch(self, train_data: torch.Tensor, bptt: int) -> Tuple[float, float]:
        """Train for one epoch."""
        self.model.train()
        total_loss = 0.0
        total_tokens = 0
        
        progress_bar = tqdm(range(0, train_data.size(0) - 1, bptt), 
                           desc="Training", leave=False)
        
        for i in progress_bar:
            # Get batch
            seq_len = min(bptt, train_data.size(0) - 1 - i)
            data = train_data[i:i+seq_len].to(self.device)
            targets = train_data[i+1:i+1+seq_len].to(self.device)
            
            self.optimizer.zero_grad()
            
            # Forward pass (for language modeling, use same data for src and tgt)
            output = self.model(data, data)
            loss = self.criterion(output.view(-1, self.vocab_size), targets.view(-1))
            
            # Backward pass
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.config.clip)
            
            self.optimizer.step()
            self.scheduler.step()
            
            # Update metrics
            total_loss += loss.item() * seq_len
            total_tokens += seq_len
            
            # Update progress bar
            current_lr = self.scheduler.get_last_lr()[0]
            progress_bar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'ppl': f'{math.exp(loss.item()):.2f}',
                'lr': f'{current_lr:.2e}'
            })
        
        avg_loss = total_loss / total_tokens
        return avg_loss, math.exp(avg_loss)
    
    def evaluate(self, data: torch.Tensor, bptt: int) -> Tuple[float, float]:
        """Evaluate the model."""
        self.model.eval()
        total_loss = 0.0
        total_tokens = 0
        
        with torch.no_grad():
            for i in range(0, data.size(0) - 1, bptt):
                seq_len = min(bptt, data.size(0) - 1 - i)
                data_batch = data[i:i+seq_len].to(self.device)
                targets = data[i+1:i+1+seq_len].to(self.device)
                
                output = self.model(data_batch, data_batch)
                loss = self.criterion(output.view(-1, self.vocab_size), targets.view(-1))
                
                total_loss += loss.item() * seq_len
                total_tokens += seq_len
        
        avg_loss = total_loss / total_tokens
        return avg_loss, math.exp(avg_loss)
    
    def train(self, train_data: torch.Tensor, val_data: torch.Tensor) -> Dict:
        """
        Main training loop with early stopping and checkpointing.
        """
        print("Starting training...")
        start_time = time.time()
        
        for epoch in range(self.config.epochs):
            epoch_start = time.time()
            
            # Train
            train_loss, train_ppl = self.train_epoch(train_data, self.config.bptt)
            
            # Validate
            val_loss, val_ppl = self.evaluate(val_data, self.config.bptt)
            
            # Update metrics
            self.train_losses.append(train_loss)
            self.val_losses.append(val_loss)
            self.train_perplexities.append(train_ppl)
            self.val_perplexities.append(val_ppl)
            
            # Check for improvement
            if val_loss < self.best_val_loss - self.config.min_delta:
                self.best_val_loss = val_loss
                self.patience_counter = 0
                self.save_checkpoint('best_model.pt')
            else:
                self.patience_counter += 1
            
            # Print epoch results
            epoch_time = time.time() - epoch_start
            print(f'Epoch {epoch+1}/{self.config.epochs} | '
                  f'Train Loss: {train_loss:.4f} | Train PPL: {train_ppl:.2f} | '
                  f'Val Loss: {val_loss:.4f} | Val PPL: {val_ppl:.2f} | '
                  f'Time: {epoch_time:.1f}s')
            
            # Early stopping
            if self.patience_counter >= self.config.patience:
                print(f'Early stopping triggered after {epoch+1} epochs')
                break
        
        total_time = time.time() - start_time
        print(f'Training completed in {total_time:.1f}s')
        
        return {
            'train_losses': self.train_losses,
            'val_losses': self.val_losses,
            'train_perplexities': self.train_perplexities,
            'val_perplexities': self.val_perplexities,
            'best_val_loss': self.best_val_loss,
            'total_time': total_time
        }
    
    def save_checkpoint(self, path: str):
        """Save model checkpoint."""
        torch.save({
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'scheduler_state_dict': self.scheduler.state_dict(),
            'config': self.config.to_dict(),
            'train_losses': self.train_losses,
            'val_losses': self.val_losses,
            'best_val_loss': self.best_val_loss
        }, path)
    
    def load_checkpoint(self, path: str):
        """Load model checkpoint."""
        checkpoint = torch.load(path, map_location=self.device)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        self.scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        self.train_losses = checkpoint.get('train_losses', [])
        self.val_losses = checkpoint.get('val_losses', [])
        self.best_val_loss = checkpoint.get('best_val_loss', float('inf'))
    
    def plot_training_curves(self, figsize: Tuple[int, int] = (15, 5)):
        """Plot training and validation curves."""
        fig, axes = plt.subplots(1, 2, figsize=figsize)
        
        # Loss curves
        epochs = range(1, len(self.train_losses) + 1)
        axes[0].plot(epochs, self.train_losses, 'b-', label='Training Loss')
        axes[0].plot(epochs, self.val_losses, 'r-', label='Validation Loss')
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('Loss')
        axes[0].set_title('Training and Validation Loss')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
        
        # Perplexity curves
        axes[1].plot(epochs, self.train_perplexities, 'b-', label='Training Perplexity')
        axes[1].plot(epochs, self.val_perplexities, 'r-', label='Validation Perplexity')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Perplexity')
        axes[1].set_title('Training and Validation Perplexity')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()


print("Attention visualization and enhanced training tools loaded successfully!")

In [ ]:
# ... (previous code remains unchanged)
import time
# Training function
def train(model, train_data, val_data, epochs, bptt, lr, clip):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.95)

    best_val_loss = float('inf')
    train_losses = []
    val_losses = []

    for epoch in range(epochs):
        model.train()
        total_loss = 0.
        start_time = time.time()

        for batch, i in enumerate(range(0, train_data.size(0) - 1, bptt)):
            data, targets = get_batch(train_data, i, bptt)
            data, targets = data.to(device), targets.to(device)

            optimizer.zero_grad()
            output = model(data, data)  # Using the same data for src and tgt in language modeling
            loss = criterion(output.view(-1, vocab_size), targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
            optimizer.step()

            total_loss += loss.item()

            if batch % 200 == 0 and batch > 0:
                cur_loss = total_loss / 200
                elapsed = time.time() - start_time
                print(f'| epoch {epoch:3d} | {batch:5d}/{len(train_data) // bptt:5d} batches | '
                      f'lr {scheduler.get_last_lr()[0]:02.5f} | ms/batch {elapsed * 1000 / 200:5.2f} | '
                      f'loss {cur_loss:5.2f} | ppl {math.exp(cur_loss):8.2f}')
                total_loss = 0
                start_time = time.time()

        val_loss = evaluate(model, val_data, bptt)
        print(f'| End of epoch {epoch:3d} | valid loss {val_loss:5.2f} | '
              f'valid ppl {math.exp(val_loss):8.2f}')

        train_losses.append(total_loss / (len(train_data) // bptt))
        val_losses.append(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), 'best_model.pt')

        scheduler.step()

    return train_losses, val_losses

# Evaluation function
def evaluate(model, data_source, bptt):
    model.eval()
    total_loss = 0.
    criterion = nn.CrossEntropyLoss()

    with torch.no_grad():
        for i in range(0, data_source.size(0) - 1, bptt):
            data, targets = get_batch(data_source, i, bptt)
            data, targets = data.to(device), targets.to(device)
            output = model(data, data)
            total_loss += criterion(output.view(-1, vocab_size), targets).item() * len(data)

    return total_loss / (len(data_source) - 1)

# Prediction function
def predict(model, text, max_len=50):
    model.eval()
    tokens = tokenizer(text)
    input_ids = torch.tensor([vocab[token] for token in tokens]).unsqueeze(0).to(device)

    with torch.no_grad():
        for _ in range(max_len):
            output = model(input_ids, input_ids)
            next_token_id = output[0, -1, :].argmax().item()
            input_ids = torch.cat([input_ids, torch.tensor([[next_token_id]]).to(device)], dim=1)

            if next_token_id == vocab['<eos>']:
                break

    predicted_tokens = [vocab.get_itos()[id] for id in input_ids[0]]
    return ' '.join(predicted_tokens)

# Training parameters
epochs = 10
bptt = 35  # sequence length
lr = 0.0001
clip = 0.25

# Train the model
print("Training the model...")
train_losses, val_losses = train(model, train_data, val_data, epochs, bptt, lr, clip)

# Plot training and validation losses
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Training Loss')
plt.plot(val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Losses')
plt.legend()
plt.savefig('loss_plot.png')
plt.close()

# Evaluate on test data
test_loss = evaluate(model, test_data, bptt)
print(f'| Test loss {test_loss:5.2f} | Test ppl {math.exp(test_loss):8.2f}')

# Example prediction
input_text = "The quick brown fox"
predicted_text = predict(model, input_text)
print(f"Input: {input_text}")
print(f"Predicted: {predicted_text}")

Training the model...
| End of epoch   0 | valid loss  8.01 | valid ppl  3024.83
| End of epoch   1 | valid loss  7.26 | valid ppl  1427.78
| End of epoch   2 | valid loss  6.95 | valid ppl  1046.39
| End of epoch   3 | valid loss  6.87 | valid ppl   966.06
| End of epoch   4 | valid loss  6.80 | valid ppl   897.41
| End of epoch   5 | valid loss  6.78 | valid ppl   879.67
| End of epoch   6 | valid loss  6.77 | valid ppl   875.21
| End of epoch   7 | valid loss  6.77 | valid ppl   869.02
| End of epoch   8 | valid loss  6.77 | valid ppl   873.11
| End of epoch   9 | valid loss  6.78 | valid ppl   879.79
| Test loss  6.93 | Test ppl  1020.01
Input: The quick brown fox
Predicted: the quick brown fox , the blue jackets ' s . the blue jackets ' s . the game ' s . the game ' s . the game ' s . the game ' s . the game . the game ' s . the game . the game . the game '


This `MultiHeadAttention` class in PyTorch implements the multi-head attention mechanism, a core component of the Transformer architecture. Below is a line-by-line breakdown of the mathematical basis behind each part of the code:

### Constructor (`__init__` method)
1. **`class MultiHeadAttention(nn.Module):`**
   - This defines a PyTorch neural network module named `MultiHeadAttention`, which represents the multi-head attention mechanism.

2. **`def __init__(self, d_model, d_k, d_v, num_heads):`**
   - The constructor initializes the `MultiHeadAttention` class with four parameters: `d_model`, `d_k`, `d_v`, and `num_heads`.
     - $d_{\text{model}}$: The dimension of the model (input/output dimensionality).
     - $d_k$: The dimensionality of the queries and keys.
     - $d_v$: The dimensionality of the values.
     - `num_heads`: The number of attention heads.

3. **`super().__init__()`**
   - Calls the constructor of the parent class `nn.Module` to initialize the module's internal states.

4. **`self.num_heads = num_heads`**
   - Stores the number of attention heads as an instance variable.

5. **`self.d_model = d_model`, `self.d_k = d_k`, `self.d_v = d_v`**
   - Stores the input model dimension ($d_{\text{model}}$), query/key dimension ($d_k$), and value dimension ($d_v$) as instance variables.

6. **`self.wq = nn.Linear(d_model, d_k * num_heads)`**
   - Defines a linear layer to project the input (`q`) from the model dimension ($d_{\text{model}}$) to the concatenated dimensions of the queries across all attention heads ($d_k \times \text{num\_heads}$).
     - This step corresponds to the creation of the query matrix $QW_i^Q$ in multi-head attention.

7. **`self.wk = nn.Linear(d_model, d_k * num_heads)`**
   - Defines a linear layer to project the input (`k`) from the model dimension ($d_{\text{model}}$) to the concatenated dimensions of the keys across all attention heads ($d_k \times \text{num\_heads}$).
     - This corresponds to the creation of the key matrix $KW_i^K$.

8. **`self.wv = nn.Linear(d_model, d_v * num_heads)`**
   - Defines a linear layer to project the input (`v`) from the model dimension ($d_{\text{model}}$) to the concatenated dimensions of the values across all attention heads ($d_v \times \text{num\_heads}$).
     - This corresponds to the creation of the value matrix $VW_i^V$.

9. **`self.wo = nn.Linear(d_v * num_heads, d_model)`**
   - Defines a linear layer that transforms the concatenated outputs from all attention heads back to the model dimension ($d_{\text{model}}$).
     - This step corresponds to applying the projection matrix $W^O$ in multi-head attention.

### Forward Pass (`forward` method)
1. **`def forward(self, q, k, v, mask=None):`**
   - This defines the forward pass of the multi-head attention module. It takes in queries (`q`), keys (`k`), values (`v`), and an optional attention mask (`mask`).

2. **`batch_size = q.size(0)`**
   - Extracts the batch size from the query tensor `q`, assuming that `q` has the shape $[\text{batch\_size}, \text{sequence\_length}, d_{\text{model}}]$.

3. **`q = self.wq(q).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)`**
   - Projects the input `q` using the `wq` linear layer to transform it from $[\text{batch\_size}, \text{sequence\_length}, d_{\text{model}}]$ to $[\text{batch\_size}, \text{sequence\_length}, \text{num\_heads} \times d_k]$.
   - Then reshapes it to $[\text{batch\_size}, \text{num\_heads}, \text{sequence\_length}, d_k]$ and transposes the dimensions to allow for multi-head parallelism.
     - This corresponds to creating multiple query matrices for different attention heads.

4. **`k = self.wk(k).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)`**
   - Similar to the above step, projects the input `k` using the `wk` linear layer, reshapes it to $[\text{batch\_size}, \text{num\_heads}, \text{sequence\_length}, d_k]$, and transposes it for multi-head parallelism.
     - This corresponds to creating multiple key matrices for different attention heads.

5. **`v = self.wv(v).view(batch_size, -1, self.num_heads, self.d_v).transpose(1, 2)`**
   - Similarly, projects the input `v` using the `wv` linear layer, reshapes it to $[\text{batch\_size}, \text{num\_heads}, \text{sequence\_length}, d_v]$, and transposes it for multi-head parallelism.
     - This corresponds to creating multiple value matrices for different attention heads.

6. **`attn_scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)`**
   - Computes the scaled dot-product attention scores by performing a matrix multiplication between `q` and the transposed `k`, resulting in attention scores of shape $[\text{batch\_size}, \text{num\_heads}, \text{sequence\_length}, \text{sequence\_length}]$.
   - Divides by the square root of $d_k$ to scale the scores, stabilizing gradients during training as described in the paper.
     - Mathematically, this corresponds to computing the similarity $ \frac{QK^T}{\sqrt{d_k}} $.

7. **`if mask is not None:`**
   - Checks if a mask is provided. Masks are typically used to prevent attention to certain positions (e.g., padding tokens or future tokens in sequence generation).

8. **`attn_scores = attn_scores.masked_fill(mask == 0, float('-inf'))`**
   - If a mask is provided, sets the attention scores of masked positions to negative infinity, effectively preventing them from contributing to the attention weights.

9. **`self.attn_probs = torch.softmax(attn_scores, dim=-1)`**
   - Applies the softmax function to the attention scores along the last dimension (across sequence length) to convert the scores into normalized attention probabilities.
     - This corresponds to applying the softmax function in the scaled dot-product attention formula.

10. **`output = torch.matmul(self.attn_probs, v)`**
    - Computes the weighted sum of the values `v` based on the attention probabilities, resulting in the attended output of shape $[\text{batch\_size}, \text{num\_heads}, \text{sequence\_length}, d_v]$.
      - This corresponds to the final step in the scaled dot-product attention formula: $ \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V $.

11. **`output = output.transpose(1, 2).contiguous().view(batch_size, -1, self.d_v * self.num_heads)`**
    - Transposes the dimensions back to $[\text{batch\_size}, \text{sequence\_length}, \text{num\_heads}, d_v]$, reshapes the output to combine the heads into a single dimension $[\text{batch\_size}, \text{sequence\_length}, \text{num\_heads} \times d_v]$, and ensures that the tensor is contiguous in memory.

12. **`return self.wo(output)`**
    - Finally, applies the `wo` linear layer to project the concatenated output back to the model dimension ($d_{\text{model}}$), producing the final output of the multi-head attention mechanism.
      - This corresponds to the application of the projection matrix $W^O$ in multi-head attention.

This class efficiently implements multi-head attention by splitting the input into multiple heads, performing scaled dot-product attention for each head, and then concatenating and projecting the results back into the model's dimensionality.


In [ ]:
import torch
import torch.nn as nn
import math

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, d_k, d_v, num_heads):
        """
        Initialize the Multi-Head Attention module.

        Args:
        - d_model: Dimensionality of the input and output (model dimension)
        - d_k: Dimensionality of keys
        - d_v: Dimensionality of values
        - num_heads: Number of parallel attention heads
        """
        super().__init__()
        self.num_heads = num_heads
        self.d_model = d_model
        self.d_k = d_k
        self.d_v = d_v

        # Linear projections for Query (Q), Key (K), and Value (V)
        self.wq = nn.Linear(d_model, d_k * num_heads)
        self.wk = nn.Linear(d_model, d_k * num_heads)
        self.wv = nn.Linear(d_model, d_v * num_heads)

        # Final output projection
        self.wo = nn.Linear(d_v * num_heads, d_model)

    def split_heads(self, x, batch_size):
        """
        Split the last dimension into (num_heads, depth).
        Transpose the result such that the shape is (batch_size, num_heads, seq_len, depth)

        Args:
        - x: Input tensor of shape (batch_size, seq_len, d_model)
        - batch_size: Size of the batch

        Returns:
        - Tensor of shape (batch_size, num_heads, seq_len, d_k) or (batch_size, num_heads, seq_len, d_v)
        """
        # Reshape to (batch_size, seq_len, num_heads, depth)
        x = x.view(batch_size, -1, self.num_heads, x.size(-1) // self.num_heads)

        # Transpose to (batch_size, num_heads, seq_len, depth)
        return x.transpose(1, 2)

    def scaled_dot_product_attention(self, q, k, v, mask=None):
        """
        Calculate the attention weights.

        Args:
        - q: query shape == (batch_size, num_heads, seq_len_q, d_k)
        - k: key shape == (batch_size, num_heads, seq_len_k, d_k)
        - v: value shape == (batch_size, num_heads, seq_len_v, d_v)
        - mask: Optional mask shape == (batch_size, 1, 1, seq_len_k)

        Returns:
        - output: shape == (batch_size, num_heads, seq_len_q, d_v)
        - attention_weights: shape == (batch_size, num_heads, seq_len_q, seq_len_k)
        """

        # Calculate attention scores
        # (batch_size, num_heads, seq_len_q, d_k) @ (batch_size, num_heads, d_k, seq_len_k)
        # -> (batch_size, num_heads, seq_len_q, seq_len_k)
        matmul_qk = torch.matmul(q, k.transpose(-2, -1))

        # Scale matmul_qk
        dk = torch.tensor(self.d_k, dtype=torch.float32)
        scaled_attention_logits = matmul_qk / torch.sqrt(dk)

        # Add the mask to the scaled tensor (if provided)
        if mask is not None:
            scaled_attention_logits += (mask * -1e9)

        # Softmax is normalized on the last axis (seq_len_k) so that the scores add up to 1
        # (batch_size, num_heads, seq_len_q, seq_len_k)
        attention_weights = torch.softmax(scaled_attention_logits, dim=-1)

        # Calculate the attention output
        # (batch_size, num_heads, seq_len_q, seq_len_k) @ (batch_size, num_heads, seq_len_v, d_v)
        # -> (batch_size, num_heads, seq_len_q, d_v)
        output = torch.matmul(attention_weights, v)

        return output, attention_weights

    def forward(self, q, k, v, mask=None):
        """
        Forward pass of the multi-head attention layer.

        Args:
        - q: Query tensor of shape (batch_size, seq_len, d_model)
        - k: Key tensor of shape (batch_size, seq_len, d_model)
        - v: Value tensor of shape (batch_size, seq_len, d_model)
        - mask: Optional mask tensor

        Returns:
        - Output tensor of shape (batch_size, seq_len, d_model)
        """
        batch_size = q.size(0)

        # Linear projections and split heads
        # (batch_size, seq_len, d_model) -> (batch_size, seq_len, d_k * num_heads)
        # -> (batch_size, num_heads, seq_len, d_k)
        q = self.split_heads(self.wq(q), batch_size)
        k = self.split_heads(self.wk(k), batch_size)
        v = self.split_heads(self.wv(v), batch_size)

        # Apply scaled dot-product attention
        # scaled_attention shape: (batch_size, num_heads, seq_len, d_v)
        # self.attention_weights shape: (batch_size, num_heads, seq_len, seq_len)
        scaled_attention, self.attention_weights = self.scaled_dot_product_attention(q, k, v, mask)

        # Concatenate heads and prepare for final linear projection
        # Step 1: Transpose the scaled attention output
        # (batch_size, num_heads, seq_len, d_v) -> (batch_size, seq_len, num_heads, d_v)
        transposed_attention = scaled_attention.transpose(1, 2)

        # Step 2: Ensure the tensor is contiguous in memory
        # Shape remains (batch_size, seq_len, num_heads, d_v)
        contiguous_attention = transposed_attention.contiguous()

        # Step 3: Reshape the tensor to concatenate the heads
        # (batch_size, seq_len, num_heads, d_v) -> (batch_size, seq_len, num_heads * d_v)
        concat_attention = contiguous_attention.view(batch_size, -1, self.d_v * self.num_heads)

        # Comments on dimensions:
        # - batch_size: number of sequences in the batch
        # - seq_len: length of each sequence (inferred with -1 in view())
        # - num_heads * d_v: total dimension across all heads
        #   This combines the separate heads into one larger representation

        # Apply final linear projection
        # (batch_size, seq_len, d_v * num_heads) -> (batch_size, seq_len, d_model)
        output = self.wo(concat_attention)

        return output

# Example usage with shape visualization
def example_usage():
    # Set up parameters
    d_model = 512
    d_k = 64
    d_v = 64
    num_heads = 8
    batch_size = 32
    seq_len = 10

    # Instantiate the MultiHeadAttention module
    mha = MultiHeadAttention(d_model, d_k, d_v, num_heads)

    # Create dummy input tensors
    q = torch.randn(batch_size, seq_len, d_model)
    k = torch.randn(batch_size, seq_len, d_model)
    v = torch.randn(batch_size, seq_len, d_model)

    print("Input shapes:")
    print(f"q: {q.shape}")
    print(f"k: {k.shape}")
    print(f"v: {v.shape}")

    # Forward pass
    output = mha(q, k, v)

    print("\nOutput shape:")
    print(f"output: {output.shape}")

    print("\nAttention weights shape:")
    print(f"attention_weights: {mha.attention_weights.shape}")

    # Visualization of internal tensor shapes
    print("\nInternal tensor shapes:")
    print(f"After linear projection (e.g., q): {mha.wq(q).shape}")
    print(f"After splitting heads (e.g., q): {mha.split_heads(mha.wq(q), batch_size).shape}")

    # Scaled dot-product attention
    scaled_attention, _ = mha.scaled_dot_product_attention(
        mha.split_heads(mha.wq(q), batch_size),
        mha.split_heads(mha.wk(k), batch_size),
        mha.split_heads(mha.wv(v), batch_size)
    )
    print(f"Scaled attention output: {scaled_attention.shape}")

    # Concatenation steps
    transposed = scaled_attention.transpose(1, 2)
    print(f"After transpose: {transposed.shape}")

    contiguous = transposed.contiguous()
    print(f"After contiguous: {contiguous.shape}")

    concatenated = contiguous.view(batch_size, -1, d_v * num_heads)
    print(f"After concatenation: {concatenated.shape}")

if __name__ == "__main__":
    example_usage()

Input shapes:
q: torch.Size([32, 10, 512])
k: torch.Size([32, 10, 512])
v: torch.Size([32, 10, 512])

Output shape:
output: torch.Size([32, 10, 512])

Attention weights shape:
attention_weights: torch.Size([32, 8, 10, 10])

Internal tensor shapes:
After linear projection (e.g., q): torch.Size([32, 10, 512])
After splitting heads (e.g., q): torch.Size([32, 8, 10, 64])
Scaled attention output: torch.Size([32, 8, 10, 64])
After transpose: torch.Size([32, 10, 8, 64])
After contiguous: torch.Size([32, 10, 8, 64])
After concatenation: torch.Size([32, 10, 512])


### Attention Mechanism Explanation with Multi-Dimensional Example

The attention mechanism is a core component in models like the Transformer, where it allows the model to focus on different parts of the input sequence based on the relevance to a given task. The mechanism operates on three key components: **Query (Q)**, **Key (K)**, and **Value (V)**, which have specific roles and dimensions that facilitate the computation of attention scores and the final output.

#### Roles of Query, Key, and Value

1. **Query (Q):** Represents what the model is trying to find or match. It acts as a "search pattern" that helps the attention mechanism determine which parts of the input are most relevant.
2. **Key (K):** Serves as an identifier or feature to determine how relevant each value is to the query. The key is compared to the query to compute attention scores, which assign weights to the values.
3. **Value (V):** Represents the actual input data (or input pattern) that will be aggregated and returned as the final output of the attention mechanism. This is the data the model will focus on based on the attention scores.

#### General Structure and Dimensions

1. **Input:**
   - Suppose we have a sequence of $n$ input tokens (e.g., words in a sentence).
   - Each input token $x_i$ is typically represented as a vector of size $d_{\text{model}}$, where $d_{\text{model}}$ is the dimensionality of the input embeddings.

2. **Query, Key, and Value Dimensions:**
   - For each token $x_i$, the model computes the Query, Key, and Value vectors through linear transformations of the input embedding.
   - These vectors usually have the following dimensions:
     - **Query (Q):** $Q \in \mathbb{R}^{n \times d_k}$
     - **Key (K):** $K \in \mathbb{R}^{n \times d_k}$
     - **Value (V):** $V \in \mathbb{R}^{n \times d_v}$
   
   Here:
   - $n$ is the number of tokens (length of the input sequence).
   - $d_k$ is the dimensionality of the Key and Query vectors.
   - $d_v$ is the dimensionality of the Value vectors.

   **Note:** Typically, $d_k = d_v$, but they can differ.

3. **Attention Score Calculation:**
   - To compute attention scores, we calculate the similarity between the Query and Key vectors.
   - If we are computing dot product attention, the score for each token is computed as $Q_i \cdot K_j$, which results in a scalar value.
   - This results in an attention score matrix of size $n \times n$ (since we compute scores for each pair of tokens).

4. **Softmax and Weighted Sum:**
   - The attention scores are normalized using softmax, yielding a weight matrix of size $n \times n$.
   - These weights are then used to compute the weighted sum of the Value vectors, producing an output of size $n \times d_v$.

#### Example with Dimensions

Let’s assume:
- **Number of tokens (n):** 4 (e.g., a sentence with 4 words).
- **Embedding dimension (d_model):** 512.
- **Query/Key dimension (d_k):** 64.
- **Value dimension (d_v):** 128.

Given this:
1. **Input Embeddings:**
   - The input sequence would be represented as a matrix $X \in \mathbb{R}^{4 \times 512}$.

2. **Query, Key, Value:**
   - The Query matrix $Q$ after transformation would be $Q \in \mathbb{R}^{4 \times 64}$.
   - The Key matrix $K$ would be $K \in \mathbb{R}^{4 \times 64}$.
   - The Value matrix $V$ would be $V \in \mathbb{R}^{4 \times 128}$.

3. **Attention Scores:**
   - The dot product of $Q$ and $K^T$ (transpose of $K$) would still yield an attention score matrix of size $4 \times 4$.

4. **Output:**
   - The final output after applying softmax and weighting the values would now be of size $4 \times 128$, corresponding to the aggregated representation of the input sequence.


In [ ]:
import numpy as np

# Define dimensions
n_tokens = 5
d_model = 4
d_k = 2
d_v = 3

# Random seed for reproducibility
np.random.seed(42)

# Input embeddings (5 tokens, each of dimension 4)
X = np.random.rand(n_tokens, d_model)

# Linear transformations for Query, Key, and Value
W_Q = np.random.rand(d_model, d_k)
W_K = np.random.rand(d_model, d_k)
W_V = np.random.rand(d_model, d_v)

# Compute Query, Key, and Value matrices
Q = X @ W_Q
K = X @ W_K
V = X @ W_V

# Compute attention scores (dot product of Q and K^T)
scores = Q @ K.T

# Apply softmax to scores for each row
def softmax(x):
    e_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e_x / np.sum(e_x, axis=-1, keepdims=True)

attention_weights = softmax(scores)

# Compute the output as a weighted sum of V
output = attention_weights @ V

print("Input Embeddings (X):\n", X)
print("\nQuery Matrix (Q):\n", Q)
print("\nKey Matrix (K):\n", K)
print("\nValue Matrix (V):\n", V)
print("\nAttention Scores:\n", scores)
print("\nAttention Weights:\n", attention_weights)
print("\nOutput:\n", output)


Input Embeddings (X):
 [[0.37454012 0.95071431 0.73199394 0.59865848]
 [0.15601864 0.15599452 0.05808361 0.86617615]
 [0.60111501 0.70807258 0.02058449 0.96990985]
 [0.83244264 0.21233911 0.18182497 0.18340451]
 [0.30424224 0.52475643 0.43194502 0.29122914]]

Query Matrix (Q):
 [[0.96028642 1.28314635]
 [0.34047628 0.56993754]
 [0.77770711 0.85818613]
 [0.69091216 0.43099109]
 [0.59460426 0.72360356]]

Key Matrix (K):
 [[1.42518579 1.35804966]
 [1.02738725 0.78917716]
 [1.72421022 0.95227038]
 [0.81108596 0.39567105]
 [0.8083695  0.74891105]]

Value Matrix (V):
 [[0.95434613 1.00483113 1.227813  ]
 [0.69204521 0.35708928 0.64949938]
 [1.13806258 0.46617365 1.27166989]
 [0.47479735 0.32972629 0.81716565]
 [0.53144897 0.5773121  0.73125755]]

Attention Scores:
 [[3.11116303 1.99921581 2.87763792 1.28657869 1.73722874]
 [1.25924544 0.79958268 1.12978732 0.50166331 0.70206317]
 [2.2738365  1.47626726 2.15815578 0.97034672 1.27137979]
 [1.5699855  1.04996267 1.60169785 0.73091984 0.88128631

# Multihead Attention Mechanism: Input, Matrix Dimensions, and Resultant Dimensions

The Multihead Attention mechanism in the Transformer model allows the model to focus on different parts of the input sequence simultaneously. Here's a detailed breakdown of the input, matrix dimensions, and resultant dimensions, including where and how the softmax is applied.

## 1. **Input**

The input to the Multihead Attention mechanism is typically a sequence of token embeddings. For natural language processing tasks, these could be word or subword embeddings.

- **Input Tensor**: $(B \times L \times d_{\text{model}})$
  - $B$: Batch size (number of sequences processed in parallel).
  - $L$: Sequence length (number of tokens in each sequence).
  - $d_{\text{model}}$: Model dimension (embedding dimension for each token).

## 2. **Linear Projections to Query, Key, and Value Matrices**

The input sequence is transformed into Query (Q), Key (K), and Value (V) matrices through learned linear transformations.

- **Query (Q)**:
  - Linear transformation using a weight matrix $W_Q \in \mathbb{R}^{d_{\text{model}} \times d_k}$.
  - **Q = X * W_Q**
  - Resulting shape: $Q = (B \times L \times d_k)$

- **Key (K)**:
  - Linear transformation using a weight matrix $W_K \in \mathbb{R}^{d_{\text{model}} \times d_k}$.
  - **K = X * W_K**
  - Resulting shape: $K = (B \times L \times d_k)$

- **Value (V)**:
  - Linear transformation using a weight matrix $W_V \in \mathbb{R}^{d_{\text{model}} \times d_v}$.
  - **V = X * W_V**
  - Resulting shape: $V = (B \times L \times d_v)$

Here, $d_k = d_v = \frac{d_{\text{model}}}{h}$, where $h$ is the number of attention heads.

## 3. **Scaled Dot-Product Attention**

### 3.1. **Attention Scores (QK$^T$)**

The attention scores are calculated as the dot product of the Query matrix $Q$ and the transpose of the Key matrix $K^T$, scaled by $\frac{1}{\sqrt{d_k}}$ to prevent large values that could destabilize the softmax.

- **Q × K$^T$**:
  - Q: $(B \times L \times d_k)$
  - K$^T$: $(B \times d_k \times L)$
  - Resulting shape: $(B \times L \times L)$

This gives us an attention score matrix of shape $(B \times L \times L)$, where each element represents the attention score between a pair of tokens in the sequence.

### 3.2. **Softmax on Attention Scores**

The softmax function is applied along the last dimension of the attention score matrix (i.e., across the sequence length $L$) to normalize the scores into probabilities. The softmax is applied separately for each token in the sequence to determine the importance of every other token in the sequence relative to it.

- **Softmax(Q × K$^T$)**:
  - Input shape: $(B \times L \times L)$
  - Output shape: $(B \times L \times L)$

Each row in the resulting attention weight matrix represents the attention distribution over all tokens for a specific token in the sequence.

### 3.3. **Weighted Sum with Value Matrix (V)**

The attention weights are used to compute a weighted sum of the Value (V) vectors, producing the final output for each head.

- **Attention Output = Attention Weights × V**:
  - Attention Weights: $(B \times L \times L)$
  - $V$: $(B \times L \times d_v)$
  - Resulting shape: $(B \times L \times d_v)$

This gives us the output for a single attention head.

## 4. **Multihead Attention**

In Multihead Attention, multiple attention heads are computed in parallel, each with its own set of Query, Key, and Value matrices. After calculating the attention outputs for all heads, these outputs are concatenated and passed through a final linear layer.

### 4.1. **Concatenation of Heads**

After calculating the attention output for each head, the outputs are concatenated along the last dimension:

- **Concatenated Output**: $(B \times L \times (h \times d_v))$
  - Since $d_v = \frac{d_{\text{model}}}{h}$, the concatenated dimension becomes $h \times d_v = d_{\text{model}}$.

### 4.2. **Final Linear Projection**

The concatenated output is passed through a final linear layer to project it back to the original model dimension:

- **Weight Matrix $W_O$**: The final output weight matrix has dimensions $W_O \in \mathbb{R}^{(h \times d_v) \times d_{\text{model}}}$.
- **Final Output Calculation**:
  - The concatenated output is multiplied by $W_O$ to project it back to $d_{\text{model}}$.
  - **Final Output = Concatenated Output * $W_O$**
  - Resulting shape: $(B \times L \times d_{\text{model}})$

## Summary of Resultant Dimensions

- **Input**: $B \times L \times d_{\text{model}}$
- **Q, K, V matrices**: $B \times L \times d_k$ (per head)
- **Attention Scores (QK$^T$)**: $B \times L \times L$
- **Attention Output (per head)**: $B \times L \times d_v$
- **Concatenated Output**: $B \times L \times d_{\text{model}}$
- **Final Multihead Attention Output**: $B \times L \times d_{\text{model}}$

The Multihead Attention mechanism enables the model to focus on different parts of the input sequence in parallel, making it a powerful tool for capturing complex relationships in the data.


In [ ]:
from IPython.display import HTML
svg_code = """<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 800 600"><defs><marker id="arrowhead" markerWidth="10" markerHeight="7" refX="0" refY="3.5" orient="auto"><polygon points="0 0, 10 3.5, 0 7" fill="#000"/></marker></defs><rect x="10" y="50" width="100" height="60" fill="#f0f0f0" stroke="#000"/><text x="60" y="85" text-anchor="middle">Input</text><text x="60" y="100" text-anchor="middle" font-size="10">B×L×d_model</text><rect x="170" y="20" width="100" height="40" fill="#e6f3ff" stroke="#000"/><text x="220" y="45" text-anchor="middle" font-size="12">Linear (Q)</text><rect x="170" y="70" width="100" height="40" fill="#e6f3ff" stroke="#000"/><text x="220" y="95" text-anchor="middle" font-size="12">Linear (K)</text><rect x="170" y="120" width="100" height="40" fill="#e6f3ff" stroke="#000"/><text x="220" y="145" text-anchor="middle" font-size="12">Linear (V)</text><rect x="330" y="20" width="80" height="40" fill="#ffe6e6" stroke="#000"/><text x="370" y="45" text-anchor="middle" font-size="12">Q</text><rect x="330" y="70" width="80" height="40" fill="#e6ffe6" stroke="#000"/><text x="370" y="95" text-anchor="middle" font-size="12">K</text><rect x="330" y="120" width="80" height="40" fill="#e6e6ff" stroke="#000"/><text x="370" y="145" text-anchor="middle" font-size="12">V</text><rect x="470" y="20" width="120" height="140" fill="#fff0e6" stroke="#000"/><text x="530" y="45" text-anchor="middle" font-size="14">Attention Heads</text><text x="530" y="65" text-anchor="middle" font-size="12">Head 1</text><text x="530" y="95" text-anchor="middle" font-size="12">Head 2</text><text x="530" y="125" text-anchor="middle" font-size="12">...</text><text x="530" y="150" text-anchor="middle" font-size="12">Head h</text><rect x="650" y="70" width="100" height="40" fill="#f0e6ff" stroke="#000"/><text x="700" y="95" text-anchor="middle" font-size="12">Concatenate</text><rect x="650" y="150" width="100" height="40" fill="#e6f3ff" stroke="#000"/><text x="700" y="175" text-anchor="middle" font-size="12">Linear</text><rect x="650" y="230" width="100" height="60" fill="#f0f0f0" stroke="#000"/><text x="700" y="260" text-anchor="middle">Output</text><text x="700" y="275" text-anchor="middle" font-size="10">B×L×d_model</text><line x1="110" y1="80" x2="170" y2="40" stroke="#000" stroke-width="2" marker-end="url(#arrowhead)"/><line x1="110" y1="80" x2="170" y2="90" stroke="#000" stroke-width="2" marker-end="url(#arrowhead)"/><line x1="110" y1="80" x2="170" y2="140" stroke="#000" stroke-width="2" marker-end="url(#arrowhead)"/><line x1="270" y1="40" x2="330" y2="40" stroke="#000" stroke-width="2" marker-end="url(#arrowhead)"/><line x1="270" y1="90" x2="330" y2="90" stroke="#000" stroke-width="2" marker-end="url(#arrowhead)"/><line x1="270" y1="140" x2="330" y2="140" stroke="#000" stroke-width="2" marker-end="url(#arrowhead)"/><line x1="410" y1="40" x2="470" y2="40" stroke="#000" stroke-width="2" marker-end="url(#arrowhead)"/><line x1="410" y1="90" x2="470" y2="90" stroke="#000" stroke-width="2" marker-end="url(#arrowhead)"/><line x1="410" y1="140" x2="470" y2="140" stroke="#000" stroke-width="2" marker-end="url(#arrowhead)"/><line x1="590" y1="90" x2="650" y2="90" stroke="#000" stroke-width="2" marker-end="url(#arrowhead)"/><line x1="700" y1="110" x2="700" y2="150" stroke="#000" stroke-width="2" marker-end="url(#arrowhead)"/><line x1="700" y1="190" x2="700" y2="230" stroke="#000" stroke-width="2" marker-end="url(#arrowhead)"/><text x="140" y="30" text-anchor="middle" font-size="10">B×L×d_model</text><text x="300" y="30" text-anchor="middle" font-size="10">B×L×d_k</text><text x="300" y="80" text-anchor="middle" font-size="10">B×L×d_k</text><text x="300" y="130" text-anchor="middle" font-size="10">B×L×d_v</text><text x="620" y="70" text-anchor="middle" font-size="10">B×L×d_v</text><text x="620" y="230" text-anchor="end" font-size="10">B×L×(h×d_v)</text></svg>"""
HTML(f"""
<div style="width:800px; height:600px;">
  {svg_code}
</div>
""")

In [ ]:
from IPython.display import HTML

svg_code = """
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 800 1000">
  <defs>
    <marker id="arrowhead" markerWidth="10" markerHeight="7" refX="0" refY="3.5" orient="auto">
      <polygon points="0 0, 10 3.5, 0 7" fill="#000"/>
    </marker>
  </defs>

  <!-- Encoder -->
  <rect x="50" y="50" width="300" height="400" fill="#f0f0f0" stroke="#000" stroke-width="2"/>
  <text x="200" y="80" text-anchor="middle" font-size="20" font-weight="bold">Encoder</text>

  <!-- Encoder Input -->
  <rect x="100" y="100" width="200" height="40" fill="#e6f3ff" stroke="#000"/>
  <text x="200" y="125" text-anchor="middle">Input Embedding</text>
  <text x="200" y="140" text-anchor="middle" font-size="10">(batch_size, seq_len, d_model)</text>

  <!-- Positional Encoding -->
  <rect x="100" y="160" width="200" height="40" fill="#ffe6e6" stroke="#000"/>
  <text x="200" y="185" text-anchor="middle">Positional Encoding</text>
  <text x="200" y="200" text-anchor="middle" font-size="10">(batch_size, seq_len, d_model)</text>

  <!-- Multi-Head Attention -->
  <rect x="100" y="220" width="200" height="40" fill="#e6ffe6" stroke="#000"/>
  <text x="200" y="245" text-anchor="middle">Multi-Head Attention</text>
  <text x="200" y="260" text-anchor="middle" font-size="10">(batch_size, seq_len, d_model)</text>

  <!-- Add & Norm -->
  <rect x="100" y="280" width="200" height="40" fill="#e6e6ff" stroke="#000"/>
  <text x="200" y="305" text-anchor="middle">Add & Norm</text>
  <text x="200" y="320" text-anchor="middle" font-size="10">(batch_size, seq_len, d_model)</text>

  <!-- Feed Forward -->
  <rect x="100" y="340" width="200" height="40" fill="#fff0e6" stroke="#000"/>
  <text x="200" y="365" text-anchor="middle">Feed Forward</text>
  <text x="200" y="380" text-anchor="middle" font-size="10">(batch_size, seq_len, d_model)</text>

  <!-- Add & Norm -->
  <rect x="100" y="400" width="200" height="40" fill="#e6e6ff" stroke="#000"/>
  <text x="200" y="425" text-anchor="middle">Add & Norm</text>
  <text x="200" y="440" text-anchor="middle" font-size="10">(batch_size, seq_len, d_model)</text>

  <!-- Decoder -->
  <rect x="450" y="50" width="300" height="600" fill="#f0f0f0" stroke="#000" stroke-width="2"/>
  <text x="600" y="80" text-anchor="middle" font-size="20" font-weight="bold">Decoder</text>

  <!-- Decoder Input -->
  <rect x="500" y="100" width="200" height="40" fill="#e6f3ff" stroke="#000"/>
  <text x="600" y="125" text-anchor="middle">Output Embedding</text>
  <text x="600" y="140" text-anchor="middle" font-size="10">(batch_size, seq_len, d_model)</text>

  <!-- Positional Encoding -->
  <rect x="500" y="160" width="200" height="40" fill="#ffe6e6" stroke="#000"/>
  <text x="600" y="185" text-anchor="middle">Positional Encoding</text>
  <text x="600" y="200" text-anchor="middle" font-size="10">(batch_size, seq_len, d_model)</text>

  <!-- Masked Multi-Head Attention -->
  <rect x="500" y="220" width="200" height="40" fill="#e6ffe6" stroke="#000"/>
  <text x="600" y="245" text-anchor="middle">Masked Multi-Head Attention</text>
  <text x="600" y="260" text-anchor="middle" font-size="10">(batch_size, seq_len, d_model)</text>

  <!-- Add & Norm -->
  <rect x="500" y="280" width="200" height="40" fill="#e6e6ff" stroke="#000"/>
  <text x="600" y="305" text-anchor="middle">Add & Norm</text>
  <text x="600" y="320" text-anchor="middle" font-size="10">(batch_size, seq_len, d_model)</text>

  <!-- Multi-Head Attention -->
  <rect x="500" y="340" width="200" height="40" fill="#e6ffe6" stroke="#000"/>
  <text x="600" y="365" text-anchor="middle">Multi-Head Attention</text>
  <text x="600" y="380" text-anchor="middle" font-size="10">(batch_size, seq_len, d_model)</text>

  <!-- Add & Norm -->
  <rect x="500" y="400" width="200" height="40" fill="#e6e6ff" stroke="#000"/>
  <text x="600" y="425" text-anchor="middle">Add & Norm</text>
  <text x="600" y="440" text-anchor="middle" font-size="10">(batch_size, seq_len, d_model)</text>

  <!-- Feed Forward -->
  <rect x="500" y="460" width="200" height="40" fill="#fff0e6" stroke="#000"/>
  <text x="600" y="485" text-anchor="middle">Feed Forward</text>
  <text x="600" y="500" text-anchor="middle" font-size="10">(batch_size, seq_len, d_model)</text>

  <!-- Add & Norm -->
  <rect x="500" y="520" width="200" height="40" fill="#e6e6ff" stroke="#000"/>
  <text x="600" y="545" text-anchor="middle">Add & Norm</text>
  <text x="600" y="560" text-anchor="middle" font-size="10">(batch_size, seq_len, d_model)</text>

  <!-- Linear -->
  <rect x="500" y="580" width="200" height="40" fill="#f0e6ff" stroke="#000"/>
  <text x="600" y="605" text-anchor="middle">Linear</text>
  <text x="600" y="620" text-anchor="middle" font-size="10">(batch_size, seq_len, vocab_size)</text>

  <!-- Softmax -->
  <rect x="500" y="640" width="200" height="40" fill="#e6fff0" stroke="#000"/>
  <text x="600" y="665" text-anchor="middle">Softmax</text>
  <text x="600" y="680" text-anchor="middle" font-size="10">(batch_size, seq_len, vocab_size)</text>

  <!-- Arrows -->
  <!-- Encoder -->
  <line x1="200" y1="140" x2="200" y2="160" stroke="#000" stroke-width="2" marker-end="url(#arrowhead)"/>
  <line x1="200" y1="200" x2="200" y2="220" stroke="#000" stroke-width="2" marker-end="url(#arrowhead)"/>
  <line x1="200" y1="260" x2="200" y2="280" stroke="#000" stroke-width="2" marker-end="url(#arrowhead)"/>
  <line x1="200" y1="320" x2="200" y2="340" stroke="#000" stroke-width="2" marker-end="url(#arrowhead)"/>
  <line x1="200" y1="380" x2="200" y2="400" stroke="#000" stroke-width="2" marker-end="url(#arrowhead)"/>

  <!-- Decoder -->
  <line x1="600" y1="140" x2="600" y2="160" stroke="#000" stroke-width="2" marker-end="url(#arrowhead)"/>
  <line x1="600" y1="200" x2="600" y2="220" stroke="#000" stroke-width="2" marker-end="url(#arrowhead)"/>
  <line x1="600" y1="260" x2="600" y2="280" stroke="#000" stroke-width="2" marker-end="url(#arrowhead)"/>
  <line x1="600" y1="320" x2="600" y2="340" stroke="#000" stroke-width="2" marker-end="url(#arrowhead)"/>
  <line x1="600" y1="380" x2="600" y2="400" stroke="#000" stroke-width="2" marker-end="url(#arrowhead)"/>
  <line x1="600" y1="440" x2="600" y2="460" stroke="#000" stroke-width="2" marker-end="url(#arrowhead)"/>
  <line x1="600" y1="500" x2="600" y2="520" stroke="#000" stroke-width="2" marker-end="url(#arrowhead)"/>
  <line x1="600" y1="560" x2="600" y2="580" stroke="#000" stroke-width="2" marker-end="url(#arrowhead)"/>
  <line x1="600" y1="620" x2="600" y2="640" stroke="#000" stroke-width="2" marker-end="url(#arrowhead)"/>

  <!-- Encoder to Decoder -->
  <line x1="300" y1="420" x2="500" y2="360" stroke="#000" stroke-width="2" marker-end="url(#arrowhead)"/>

  <!-- Output -->
  <rect x="500" y="700" width="200" height="40" fill="#f0f0f0" stroke="#000"/>
  <text x="600" y="725" text-anchor="middle">Output Probabilities</text>
  <text x="600" y="740" text-anchor="middle" font-size="10">(batch_size, seq_len, vocab_size)</text>

  <line x1="600" y1="680" x2="600" y2="700" stroke="#000" stroke-width="2" marker-end="url(#arrowhead)"/>
</svg>
"""

HTML(f"""
<div style="width:800px; height:1000px;">
  {svg_code}
</div>
""")

In [ ]:
from IPython.display import HTML

svg_code = """
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 800 1000">
  <!-- Paste the entire SVG code here -->
</svg>
"""

HTML(f"""
<div style="width:800px; height:1000px;">
  {svg_code}
</div>
""")